# Notebook 3 — Spacing–speed mechanism, policy scenarios and manuscript tables

Run from the `notebooks/` folder. Inputs come from `../data/raw/`; everything this notebook makes is written to `../outputs/`
(`data/`, `tables/`, `figures/`). Figures are PNG at 600 dpi in Cambria with the house colour scheme of `style.py`; tables are Excel in Cambria.
Run the notebooks in order: **1 → 2 → 3**.

Needs notebooks 1 and 2.

| Step | Content |
|---|---|
| 4 | PR–BTW headway contrast split exactly into spacing and speed parts |
| 5 | E-rickshaw speed limits (S.R.O. No. 384-Law/2026): 15 km/h zone limit and 25 km/h general limit; exposure to short headways |
| 6 | Dedicated BTW lane at 25 km/h |
| 7 | Sensitivity of the main results to the 0.5–5 s extraction window |
| Tables | All manuscript and appendix tables in one workbook, `outputs/Manuscript_Tables.xlsx` |

## Step 4 — Spacing–speed decomposition of the PR–BTW headway contrast (by site)

**Question:** within each site, PR followers keep longer time headways than BTW followers. Is that because they keep a larger **gap in space**, or because they are **slower** and need more time to cover a similar gap?

**Identity:** for every event, the spacing is $S_i = H_i\,v_i$, where $v_i$ is the follower's space-mean speed (m/s). $S_i$ is the distance the follower covers in its headway (front-to-front space headway at the follower's speed). Then exactly,

$$\log H_i = \log S_i - \log v_i .$$

**Contrast at the same leader mix:** PR and BTW followers meet different leader mixes. Within each site, the mean log of each quantity is computed per target class and leader class, then averaged over leader classes with one common set of weights, the site's pooled leader shares (direct standardisation). Because means are linear, the standardised contrast splits **exactly**:

$$\underbrace{\Delta\overline{\log H}}_{\text{headway}} = \underbrace{\Delta\overline{\log S}}_{\text{spacing part}} \;+\; \underbrace{\left(-\Delta\overline{\log v}\right)}_{\text{speed part}},\qquad \Delta = \text{PR} - \text{BTW}.$$

Exponentiated, these are ratios of geometric means: headway ratio = spacing ratio × (BTW speed ÷ PR speed). The **speed share** is $-\Delta\overline{\log v}\,/\,\Delta\overline{\log H}$.

**Uncertainty:** 95% percentile intervals from a stratified bootstrap (resampling events within site × target class × leader class cells), B = 4,000.

**Checks:**
- The identity holds to machine precision for every event.
- A regression version, OLS of each log quantity on target class plus leader dummies, gives the same exact split (shared design matrix).
- Per-leader contrasts show whether the split is the same behind every leader class.

**Scope:** the sites are never merged. Every quantity is conditional on following within the 0.5–5 s extraction window, as in steps 1–3.

**Outputs:** `outputs/tables/04_spacing_speed_decomposition.xlsx`, and figures `decomposition_contrast.png`, `spacing_speed_scatter.png`.

In [ ]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path.cwd()))
from style import PAL, use_style
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from openpyxl import load_workbook
from openpyxl.styles import Font

HERE = Path.cwd()
DATA = HERE.parent / "outputs" / "data" / "data04.xlsx"
TAB = HERE.parent / "outputs" / "tables"; TAB.mkdir(parents=True, exist_ok=True)
FIG = HERE.parent / "outputs" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
FONT = "Cambria"
plt.rcParams.update({"font.family": FONT, "mathtext.fontset": "custom", "mathtext.rm": FONT,
                     "mathtext.it": FONT + ":italic", "mathtext.bf": FONT + ":bold", "font.size": 9})
def save_fig(fig, name):
    p = FIG / f"{name}.png"; p.unlink(missing_ok=True); fig.savefig(p, dpi=600, bbox_inches="tight"); print("saved", p.name)
def cambria_xlsx(path):
    wb = load_workbook(path)
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for c in row:
                c.font = Font(name=FONT, bold=(c.row == 1), size=11)
    wb.save(path)

### Settings

In [ ]:
SITES = ["Shahjahanpur", "Tikatuli"]
TARGETS = ["PR", "BTW"]
LEADERS = ["4W", "BTW", "CNG", "MT_2W", "NMT"]
B = 4000; SEED = 20261107
QTY = {"logH": "Headway", "logS": "Spacing part", "neg_logv": "Speed part"}
QCOL = PAL["component"]
TCOL = PAL["target"]

### Data and the identity

In [ ]:
d = pd.read_excel(DATA, sheet_name="data04")
print("data04:", d.shape, " sha256:", hashlib.sha256(DATA.read_bytes()).hexdigest()[:16])
d["v_ms"] = d.Target_Speed_kmh / 3.6
d["S_m"] = d.Time_Headway_s * d.v_ms
d["logH"], d["logS"], d["logv"] = np.log(d.Time_Headway_s), np.log(d.S_m), np.log(d.v_ms)
d["neg_logv"] = -d.logv
err = np.abs(d.logH - (d.logS - d.logv)).max(); print("max |log H - (log S - log v)| =", err); assert err < 1e-12
desc = (d.groupby(["Site", "Target_Class"])
        .agg(n=("Obs_ID", "size"), gm_headway_s=("logH", lambda s: np.exp(s.mean())), gm_spacing_m=("logS", lambda s: np.exp(s.mean())),
             gm_speed_kmh=("logv", lambda s: 3.6 * np.exp(s.mean())), median_spacing_m=("S_m", "median"),
             p10_spacing_m=("S_m", lambda s: s.quantile(.1)), p90_spacing_m=("S_m", lambda s: s.quantile(.9)))
        .reindex(pd.MultiIndex.from_product([SITES, TARGETS])).reset_index().rename(columns={"level_0": "Site", "level_1": "Target_Class"}))
print(desc.round(3).to_string(index=False))

### Standardised contrast and its exact split

In [ ]:
def cell_means(g):
    return g.groupby(["Target_Class", "Leader_Class"])[list(QTY)].mean()

def standardised(g, w):
    m = cell_means(g)
    out = {}
    for t in TARGETS:
        mt = m.loc[t].reindex(LEADERS)
        out[t] = (mt.mul(w, axis=0)).sum()
    return out["PR"] - out["BTW"], out

rng = np.random.default_rng(SEED)
rows, boot_store = [], {}
for site in SITES:
    g = d[d.Site == site]
    w = g.Leader_Class.value_counts(normalize=True).reindex(LEADERS)
    assert (g.groupby(["Target_Class", "Leader_Class"]).size().unstack().reindex(columns=LEADERS) > 0).all().all()
    est, lev = standardised(g, w)
    # stratified bootstrap within target x leader cells; weights held at the site's observed leader shares
    idx = [np.flatnonzero(((g.Target_Class == t) & (g.Leader_Class == c)).to_numpy()) for t in TARGETS for c in LEADERS]
    vals = g[list(QTY)].to_numpy(); keys = [(t, c) for t in TARGETS for c in LEADERS]
    bs = np.empty((B, len(QTY)))
    for b in range(B):
        mean = {k: vals[rng.choice(ix, size=len(ix), replace=True)].mean(axis=0) for k, ix in zip(keys, idx)}
        pr = sum(w[c] * mean[("PR", c)] for c in LEADERS); bt = sum(w[c] * mean[("BTW", c)] for c in LEADERS)
        bs[b] = pr - bt
    boot_store[site] = bs
    share_b = bs[:, 2] / bs[:, 0]
    for j, q in enumerate(QTY):
        lo, hi = np.percentile(bs[:, j], [2.5, 97.5])
        rows.append({"Site": site, "Component": QTY[q], "key": q, "log_contrast": est[q], "log_lo95": lo, "log_hi95": hi,
                     "ratio_PR_over_BTW": np.exp(est[q]), "ratio_lo95": np.exp(lo), "ratio_hi95": np.exp(hi),
                     "pct": 100 * (np.exp(est[q]) - 1), "pct_lo95": 100 * (np.exp(lo) - 1), "pct_hi95": 100 * (np.exp(hi) - 1),
                     "share_of_headway_contrast": est[q] / est["logH"] if q != "logH" else 1.0,
                     "share_lo95": np.percentile(bs[:, j] / bs[:, 0], 2.5) if q != "logH" else np.nan,
                     "share_hi95": np.percentile(bs[:, j] / bs[:, 0], 97.5) if q != "logH" else np.nan})
    print(site, " exact split check:", round(est["logH"] - est["logS"] - est["neg_logv"], 15))
dec = pd.DataFrame(rows)
print(dec.drop(columns="key").round(4).to_string(index=False))

### Check: the regression version gives the same split

OLS of each log quantity on a PR indicator plus leader dummies, within site. The three regressions share one design matrix, so the PR coefficients add up exactly: $b_H = b_S + b_{-v}$. This contrast weights leaders by precision instead of by the site's shares, so it is close to, but not identical with, the standardised contrast.

In [ ]:
ols = []
for site in SITES:
    g = d[d.Site == site]
    X = np.column_stack([np.ones(len(g)), (g.Target_Class == "PR").to_numpy(float)] +
                        [(g.Leader_Class == c).to_numpy(float) for c in LEADERS[1:]])
    coef = {q: np.linalg.lstsq(X, g[q].to_numpy(), rcond=None)[0][1] for q in QTY}
    ols.append({"Site": site, **{f"b_{q}": v for q, v in coef.items()}, "split_error": coef["logH"] - coef["logS"] - coef["neg_logv"],
                "speed_share": coef["neg_logv"] / coef["logH"], "headway_ratio": np.exp(coef["logH"])})
ols = pd.DataFrame(ols); print(ols.round(5).to_string(index=False))
assert ols.split_error.abs().max() < 1e-10

### Per-leader contrasts

In [ ]:
pl = []
for site in SITES:
    m = cell_means(d[d.Site == site]); n = d[d.Site == site].groupby(["Target_Class", "Leader_Class"]).size()
    for c in LEADERS:
        diff = m.loc[("PR", c)] - m.loc[("BTW", c)]
        pl.append({"Site": site, "Leader": c, "n_PR": int(n[("PR", c)]), "n_BTW": int(n[("BTW", c)]),
                   "headway_ratio": np.exp(diff.logH), "spacing_ratio": np.exp(diff.logS), "speed_ratio_BTW_over_PR": np.exp(diff.neg_logv),
                   "speed_share": diff.neg_logv / diff.logH})
pl = pd.DataFrame(pl); print(pl.round(3).to_string(index=False))

### Figure: the PR–BTW headway contrast split into spacing and speed

In [ ]:
# Waterfall on the log scale, where the split is additive: the speed part starts at 0, the spacing part starts where
# the speed part ends, and together they reach the headway bar. Tick labels are shown as % (PR relative to BTW).
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.9), sharex=True)
Y = {"logH": 2, "logS": 1, "neg_logv": 0}
for ax, site, tag in zip(axes, SITES, "ab"):
    g = dec[dec.Site == site].set_index("key")
    sp = g.loc["neg_logv", "log_contrast"]
    start = {"logH": 0.0, "neg_logv": 0.0, "logS": sp}
    for q in QTY:
        v, lo, hi = g.loc[q, "log_contrast"], g.loc[q, "log_lo95"], g.loc[q, "log_hi95"]
        y, s0 = Y[q], start[q]
        ax.barh(y, v, left=s0, color=QCOL[q], alpha=.85, height=.55)
        ax.plot([s0 + lo, s0 + hi], [y, y], color="k", lw=1.1)
        ax.text(max(s0 + hi, s0 + v) + 0.02, y, f"{g.loc[q, 'pct']:+.0f}%", va="center", fontsize=8)
    ax.plot([sp, sp], [Y["neg_logv"] + .275, Y["logS"] - .275], color="0.35", lw=.8, ls=":")
    tot = g.loc["logH", "log_contrast"]
    ax.plot([tot, tot], [Y["logS"] + .275, Y["logH"] - .275], color="0.35", lw=.8, ls=":")
    share = g.loc["neg_logv", "share_of_headway_contrast"]
    ax.set_yticks([2, 1, 0])
    ax.set_yticklabels(["Headway\n(PR ÷ BTW)", "Spacing part", "Speed part\n(BTW ÷ PR speed)"] if tag == "a" else [])
    ax.axvline(0, color="0.4", lw=.7); ax.grid(axis="x", alpha=.25, lw=.5)
    ax.set_title(f"({tag}) {site}\nspeed part = {100 * share:.0f}% of the log headway contrast", loc="left", fontsize=8.5)
PCT = [0, 20, 40, 60, 80]
ax.set_xticks(np.log1p(np.array(PCT) / 100)); ax.set_xticklabels([f"+{p}%" if p else "0" for p in PCT])
ax.set_xlim(np.log(0.92), np.log(2.2))
fig.supxlabel("PR relative to BTW, same leader mix (log scale; segments add up; geometric means, 95% bootstrap CI)", fontsize=8.5)
fig.tight_layout(); save_fig(fig, "decomposition_contrast"); plt.show()

### Figure: spacing against speed

Each point is one following event. The grey lines are constant headway ($S = H\,v$ for H = 1, 2, 3, 4 s). If PR and BTW followers kept the same spacing at all speeds, the two clouds would spread along the same horizontal band, and the headway difference would come from where they sit on the speed axis.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.3), sharey=True)
VMAX, SMAX = 38, 30
vgrid = np.linspace(3, VMAX, 100)
for ax, site, tag in zip(axes, SITES, "ab"):
    for h in (1, 2, 3, 4):
        ax.plot(vgrid, h * vgrid / 3.6, color="0.75", lw=.7, zorder=0)
        v_end = min(VMAX, SMAX * 3.6 / h)                        # where the line leaves the plotting area
        ax.text(v_end - 0.4, h * v_end / 3.6 - 0.4, f"H = {h} s", fontsize=6.5, color="0.45", va="top", ha="right")
    for t in TARGETS:
        g = d[(d.Site == site) & (d.Target_Class == t)]
        ax.scatter(g.Target_Speed_kmh, g.S_m, s=5, alpha=.45, color=TCOL[t], lw=0, label=f"{t} (n={len(g)})")
        gm_v, gm_s = 3.6 * np.exp(g.logv.mean()), np.exp(g.logS.mean())
        ax.plot(gm_v, gm_s, marker="D", ms=6, color=TCOL[t], mec="k", mew=.8)
    ax.set_xlim(3, VMAX); ax.set_ylim(0, SMAX); ax.grid(alpha=.2, lw=.5)
    ax.set_title(f"({tag}) {site}", loc="left", fontsize=8.5); ax.set_xlabel("Follower speed (km/h)")
    ax.legend(frameon=False, fontsize=7, loc="upper left")
axes[0].set_ylabel("Spacing S = H · v (m)")
fig.tight_layout(); save_fig(fig, "spacing_speed_scatter"); plt.show()

### Export

In [ ]:
out = TAB / "04_spacing_speed_decomposition.xlsx"
with pd.ExcelWriter(out) as xw:
    dec.drop(columns="key").to_excel(xw, sheet_name="decomposition", index=False)
    desc.to_excel(xw, sheet_name="by_site_target", index=False)
    pl.to_excel(xw, sheet_name="per_leader", index=False)
    ols.to_excel(xw, sheet_name="regression_check", index=False)
    pd.DataFrame({"Parameter": ["spacing_definition", "speed", "standardisation_weights", "bootstrap", "B", "SEED", "data04_sha256"],
                  "Value": ["S = H x v_follower (m)", "follower space-mean speed over the segment (m/s)", "site pooled leader shares",
                            "stratified within site x target x leader", B, SEED, hashlib.sha256(DATA.read_bytes()).hexdigest()]}
                 ).to_excel(xw, sheet_name="settings", index=False)
cambria_xlsx(out); print("wrote", out.name)

## Step 5 — E-rickshaw speed-limit scenarios (by site)

**Policy:** City Corporation Areas Three-Wheeler Slow-Speed Battery-Operated Rickshaw (E-Rickshaw) Operation Rules, 2026. S.R.O. No. 384-Law/2026, Bangladesh Gazette (extraordinary), 29 October 2026, Local Government Division.
- Rule 2(ঘ) limits the e-rickshaw by design to **25 km/h**.
- Rule 8(4) sets **15 km/h** near schools, hospitals, places of worship and crowded places.
- The rules apply to e-rickshaws (BTW) only, not to pedal rickshaws (PR).

**Scenarios:**
- **15 km/h (zone limit):** both study segments were crowded during data collection, so this is the limit that applies at both sites.
- **25 km/h (general limit):** the e-rickshaw design limit anywhere in a city corporation area.

**Sites (context for reading the results):** Shahjahanpur is signal-controlled and narrower, with prominent on-street parking. Tikatuli is a wider, single-lane segment without signal control and is the more crowded of the two.

**What a cap does to an observed BTW following event.** The cap only binds when the BTW was faster than the cap ($v > c$). What happens then depends on the leader:

| Case | Condition | Outcome under the cap |
|---|---|---|
| 1. Not affected | $v \le c$ | headway unchanged |
| 2. Following breaks up | $v > c$, leader is **not** an e-rickshaw and travels faster than the cap ($v_L > c$) | the leader keeps its speed and the capped BTW falls behind; the gap keeps opening, so there is no steady following headway |
| 3. Following continues at the capped speed | $v > c$, and the leader is a BTW (capped as well) or already at or below the cap ($v_L \le c$) | the BTW follows at $v' = c$ with the same spacing, $S' = S$, so $H' = H\,v/c$ |

Case 3 holds the gap in space constant (step 4: PR and BTW keep similar spacing, and the headway contrast is mostly speed). A case-3 event whose new headway exceeds 5 s also leaves the following window.

**PR followers** are not capped. A PR behind a BTW leader that was faster than the cap now has a slower leader. These events are counted as **indirectly affected**. No headway is modelled for them, because leader speed had no clear effect on PR headways (step 3).

**Leader-speed sensitivity (Shahjahanpur BTW):** BTW followers at Shahjahanpur keep longer headways behind faster leaders (leader-adjusted time ratio ≈ 1.09 per 5 km/h, step 3). For case-3 events behind a capped BTW leader, the headway is also multiplied by $\text{TR}^{(v_L'-v_L)/5}$.

**Reported per site × cap:**
- the shares of the three cases;
- for the BTW events that keep following (cases 1 and 3 within 5 s), the median, mean and $P(H<2\ \text{s})$, compared with the observed BTW and PR distributions.

**Two effects in the change for the events that keep following.** The change from the observed BTW distribution mixes:
- a **selection effect**: the events that break up are those behind fast leaders, mostly four-wheelers, which had longer headways;
- a **speed effect**: case-3 events now run at the capped speed.

Each change is split exactly (in log-median and in percentage points): *selection* compares the surviving events at their observed headways with all observed events; *speed* compares the surviving events after and before the cap.

These are counterfactual transformations of the observed events, not a traffic simulation.

**Uncertainty:** 95% percentile intervals from a bootstrap that resamples events within site × target × leader-class cells (B = 2,000).

**Outputs:** `outputs/tables/05_speed_cap_scenarios.xlsx`, and figures `speed_cap_outcome_shares.png`, `speed_cap_headway_ecdf.png`, `speed_cap_short_headway_exposure.png`.

In [ ]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path.cwd()))
from style import PAL, use_style
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from openpyxl import load_workbook
from openpyxl.styles import Font

HERE = Path.cwd()
DATA = HERE.parent / "outputs" / "data" / "data04.xlsx"
TAB = HERE.parent / "outputs" / "tables"; TAB.mkdir(parents=True, exist_ok=True)
FIG = HERE.parent / "outputs" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
FONT = "Cambria"
plt.rcParams.update({"font.family": FONT, "mathtext.fontset": "custom", "mathtext.rm": FONT,
                     "mathtext.it": FONT + ":italic", "mathtext.bf": FONT + ":bold", "font.size": 9})
def save_fig(fig, name):
    p = FIG / f"{name}.png"; p.unlink(missing_ok=True); fig.savefig(p, dpi=600, bbox_inches="tight"); print("saved", p.name)
def cambria_xlsx(path):
    wb = load_workbook(path)
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for c in row:
                c.font = Font(name=FONT, bold=(c.row == 1), size=11)
    wb.save(path)

### Settings

In [ ]:
SITES = ["Shahjahanpur", "Tikatuli"]
CAPS = {15: "15 km/h (zone limit, rule 8(4))", 25: "25 km/h (general limit, rule 2(ঘ))"}
LEADERS = ["4W", "BTW", "CNG", "MT_2W", "NMT"]
UPPER = 5.0                        # s, upper edge of the extraction window
B = 2000; SEED = 20261107
CASES = {1: "Not affected (v ≤ cap)", 2: "Following breaks up (faster uncapped leader)", 3: "Follows at the capped speed"}
CASE_COL = PAL["outcome"]
CAP_COL = PAL["cap"]
_tr = pd.read_excel(TAB / "03_covariate_time_ratios.xlsx", sheet_name="time_ratios")
_row = _tr[(_tr.Site == "Shahjahanpur") & (_tr.Class == "BTW") & (_tr.Model == "leader_adjusted") & (_tr.Variable == "Leader_speed_5kmh")].iloc[0]
TR_LS = {"est": float(_row.TR), "lo": float(_row.TR_lo95), "hi": float(_row.TR_hi95)}
print("Shahjahanpur BTW leader-speed TR per 5 km/h:", {k: round(v, 4) for k, v in TR_LS.items()})

### Data

In [ ]:
d = pd.read_excel(DATA, sheet_name="data04")
print("data04:", d.shape, " sha256:", hashlib.sha256(DATA.read_bytes()).hexdigest()[:16])
speed = d.groupby(["Site", "Target_Class"]).Target_Speed_kmh.describe(percentiles=[.1, .5, .9]).round(2).reset_index()
print(speed.to_string(index=False))

### Scenario engine

In [ ]:
def classify(g, cap):
    v, vl = g.Target_Speed_kmh.to_numpy(), g.Leading_Speed_kmh.to_numpy()
    lead_btw = (g.Leader_Class == "BTW").to_numpy()
    case = np.ones(len(g), int)
    binds = v > cap
    case[binds & ~lead_btw & (vl > cap)] = 2
    case[binds & (lead_btw | (vl <= cap))] = 3
    return case

def btw_scenario(g, cap, leader_tr=None):
    # returns the case of each event and the new headway (NaN where following breaks up)
    H, v, vl = g.Time_Headway_s.to_numpy(), g.Target_Speed_kmh.to_numpy(), g.Leading_Speed_kmh.to_numpy()
    case = classify(g, cap)
    Hn = H.copy().astype(float)
    m3 = case == 3
    Hn[m3] = H[m3] * v[m3] / cap
    if leader_tr is not None:
        lb = m3 & (g.Leader_Class == "BTW").to_numpy() & (vl > cap)
        Hn[lb] = Hn[lb] * leader_tr ** ((cap - vl[lb]) / 5.0)
    Hn[case == 2] = np.nan
    return case, Hn

def summarise(case, Hn, H0):
    keep = np.isfinite(Hn) & (Hn <= UPPER)
    h = Hn[keep]
    h_sel = H0[keep]                                       # same surviving events, headways as observed
    sel = {"median_selection_only_s": np.median(h_sel), "P(H<2)_selection_only": (h_sel < 2).mean(),
           "d_median_pct_selection": 100 * (np.median(h_sel) / np.median(H0) - 1),
           "d_median_pct_speed": 100 * (np.median(h) / np.median(h_sel) - 1),
           "d_P(H<2)_pp_selection": 100 * ((h_sel < 2).mean() - (H0 < 2).mean()),
           "d_P(H<2)_pp_speed": 100 * ((h < 2).mean() - (h_sel < 2).mean())}
    return {**sel, **exposure(case, Hn, H0), "share_case1": (case == 1).mean(), "share_case2": (case == 2).mean(), "share_case3": (case == 3).mean(),
            "share_case3_above_5s": ((case == 3) & (Hn > UPPER)).mean(), "share_still_following": keep.mean(),
            "n_still_following": int(keep.sum()), "median_s": np.median(h), "mean_s": h.mean(), "P(H<2)": (h < 2).mean(),
            "d_median_pct": 100 * (np.median(h) / np.median(H0) - 1), "d_P(H<2)_pp": 100 * ((h < 2).mean() - (H0 < 2).mean())}

THRESH = [2.0, 1.5]                # s, short-headway thresholds for exposure

def exposure(case, Hn, H0):
    # exposure to short headways, per observed BTW event and among the events still following.
    # Accounting per threshold t (counts per 100 observed events):
    #   observed short = removed by break-up + lifted above t by the slower speed + still short
    N = len(H0); keep = np.isfinite(Hn) & (Hn <= UPPER); out = {}
    for t in THRESH:
        k = f"{t:g}"
        obs_short = H0 < t
        still = keep & (Hn < t)
        removed = (case == 2) & obs_short
        lifted = obs_short & ~still & ~removed
        out.update({f"exp{k}_obs_per100": 100 * obs_short.mean(), f"exp{k}_after_per100": 100 * still.mean(),
                    f"exp{k}_removed_breakup_per100": 100 * removed.mean(), f"exp{k}_lifted_speed_per100": 100 * lifted.mean(),
                    f"exp{k}_change_pct": 100 * (still.sum() / max(obs_short.sum(), 1) - 1),
                    f"exp{k}_share_among_following": still.sum() / max(keep.sum(), 1)})
    return out

def pr_indirect(p, cap):
    affected = (p.Leader_Class == "BTW").to_numpy() & (p.Leading_Speed_kmh.to_numpy() > cap)
    return {"PR_share_behind_capped_BTW_leader": affected.mean(),
            "PR_share_behind_capped_BTW_leader_and_PR_faster_than_cap": (affected & (p.Target_Speed_kmh.to_numpy() > cap)).mean()}

def strat_resample(g, rng):
    parts = [g[g.Leader_Class == c] for c in LEADERS]
    return pd.concat([q.iloc[rng.integers(0, len(q), len(q))] for q in parts], ignore_index=True)

### Scenarios with bootstrap intervals

In [ ]:
rng = np.random.default_rng(SEED)
rows = []
for s in SITES:
    g = d[(d.Site == s) & (d.Target_Class == "BTW")].reset_index(drop=True)
    p = d[(d.Site == s) & (d.Target_Class == "PR")].reset_index(drop=True)
    est = {}
    for cap in CAPS:
        case, Hn = btw_scenario(g, cap)
        est[cap] = {**summarise(case, Hn, g.Time_Headway_s.to_numpy()), **pr_indirect(p, cap)}
    boot = {cap: [] for cap in CAPS}
    for _ in range(B):
        gb, pb = strat_resample(g, rng), strat_resample(p, rng)
        for cap in CAPS:
            case, Hn = btw_scenario(gb, cap)
            boot[cap].append({**summarise(case, Hn, gb.Time_Headway_s.to_numpy()), **pr_indirect(pb, cap)})
    obs = g.Time_Headway_s.to_numpy(); pr_h = p.Time_Headway_s.to_numpy()
    rows.append({"Site": s, "Cap_kmh": "observed", "Scenario": "observed (no cap)", "share_case1": 1.0, "share_case2": 0.0, "share_case3": 0.0,
                 "share_case3_above_5s": 0.0, "share_still_following": 1.0, "n_still_following": len(obs), "median_s": np.median(obs),
                 "mean_s": obs.mean(), "P(H<2)": (obs < 2).mean(), "d_median_pct": 0.0, "d_P(H<2)_pp": 0.0,
                 "PR_median_s": np.median(pr_h), "PR_P(H<2)": (pr_h < 2).mean(), "n_BTW": len(g), "n_PR": len(p),
                 **exposure(np.ones(len(obs), int), obs.astype(float), obs)})
    for cap in CAPS:
        bb = pd.DataFrame(boot[cap]); r = {"Site": s, "Cap_kmh": cap, "Scenario": CAPS[cap], **est[cap]}
        for k in ["share_case1", "share_case2", "share_case3", "share_still_following", "median_s", "P(H<2)", "d_median_pct", "d_P(H<2)_pp",
                  "d_median_pct_selection", "d_median_pct_speed", "d_P(H<2)_pp_selection", "d_P(H<2)_pp_speed",
                  "PR_share_behind_capped_BTW_leader"] + \
                 [f"exp{t:g}_{m}" for t in THRESH for m in ("after_per100", "removed_breakup_per100", "lifted_speed_per100",
                                                             "change_pct", "share_among_following")]:
            r[k + "_lo95"], r[k + "_hi95"] = np.percentile(bb[k], [2.5, 97.5])
        r.update({"PR_median_s": np.median(pr_h), "PR_P(H<2)": (pr_h < 2).mean(), "n_BTW": len(g), "n_PR": len(p)})
        rows.append(r)
scen = pd.DataFrame(rows)
show = ["Site", "Cap_kmh", "share_case1", "share_case2", "share_case2_lo95", "share_case2_hi95", "share_case3", "share_case3_above_5s",
        "share_still_following", "median_s", "d_median_pct", "d_median_pct_lo95", "d_median_pct_hi95", "P(H<2)", "d_P(H<2)_pp",
        "d_P(H<2)_pp_lo95", "d_P(H<2)_pp_hi95", "PR_median_s", "PR_P(H<2)", "PR_share_behind_capped_BTW_leader"]
print(scen.reindex(columns=show).round(3).to_string(index=False))
print()
split = ["Site", "Cap_kmh", "d_median_pct", "d_median_pct_selection", "d_median_pct_speed", "d_median_pct_speed_lo95", "d_median_pct_speed_hi95",
         "d_P(H<2)_pp", "d_P(H<2)_pp_selection", "d_P(H<2)_pp_speed", "d_P(H<2)_pp_speed_lo95", "d_P(H<2)_pp_speed_hi95"]
print(scen[scen.Cap_kmh != "observed"].reindex(columns=split).round(3).to_string(index=False))

### Exposure to short headways (< 2 s and < 1.5 s)

Exposure is counted **per 100 observed BTW following events**, so the scenarios share one denominator. An event whose following breaks up no longer runs at a short headway behind that leader and is not counted. Each observed short-headway event ends in one of three states, and these add up exactly:

*observed short = removed by break-up + lifted above the threshold by the slower speed + still short.*

The share **among the events still following** is also given; it answers a different question (how close the remaining followers run). The PR benchmark is the observed PR share.

In [ ]:
pr_exp = {s: {t: 100 * (d[(d.Site == s) & (d.Target_Class == "PR")].Time_Headway_s < t).mean() for t in THRESH} for s in SITES}
ex_rows = []
for _, r in scen.iterrows():
    for t in THRESH:
        k = f"{t:g}"
        row = {"Site": r.Site, "Cap_kmh": r.Cap_kmh, "Threshold_s": t, "observed_per100": r[f"exp{k}_obs_per100"],
               "after_per100": r[f"exp{k}_after_per100"], "removed_by_breakup_per100": r[f"exp{k}_removed_breakup_per100"],
               "lifted_by_speed_per100": r[f"exp{k}_lifted_speed_per100"], "change_in_exposed_events_pct": r[f"exp{k}_change_pct"],
               "share_among_still_following": r[f"exp{k}_share_among_following"], "PR_observed_per100": pr_exp[r.Site][t]}
        for m, col in [("after_per100", "after_per100"), ("change_pct", "change_in_exposed_events_pct"),
                       ("share_among_following", "share_among_still_following"),
                       ("removed_breakup_per100", "removed_by_breakup_per100"), ("lifted_speed_per100", "lifted_by_speed_per100")]:
            row[col + "_lo95"] = r.get(f"exp{k}_{m}_lo95", np.nan); row[col + "_hi95"] = r.get(f"exp{k}_{m}_hi95", np.nan)
        ex_rows.append(row)
expo = pd.DataFrame(ex_rows)
assert np.allclose(expo.observed_per100, expo.after_per100 + expo.removed_by_breakup_per100 + expo.lifted_by_speed_per100)
print(expo[["Site", "Cap_kmh", "Threshold_s", "observed_per100", "after_per100", "after_per100_lo95", "after_per100_hi95",
            "removed_by_breakup_per100", "lifted_by_speed_per100", "change_in_exposed_events_pct", "share_among_still_following",
            "share_among_still_following_lo95", "share_among_still_following_hi95", "PR_observed_per100"]].round(3).to_string(index=False))

### Figure: exposure to short headways per 100 observed BTW following events

In [ ]:
SEG = [("after_per100", "Still at a short headway", PAL["exposure"]["still"]), ("lifted_by_speed_per100", "Lifted above the threshold by the slower speed", PAL["exposure"]["lifted_speed"]),
       ("removed_by_breakup_per100", "Removed: following breaks up", PAL["exposure"]["removed"])]
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.4), sharey=True)
xlab = ["Obs.", "25", "15"]
for ax, s, tag in zip(axes, SITES, "ab"):
    for j, t in enumerate(THRESH):
        sub = expo[(expo.Site == s) & (expo.Threshold_s == t)].set_index("Cap_kmh").loc[["observed", 25, 15]]
        x0 = j * 4.0
        for i, (cap, r) in enumerate(sub.iterrows()):
            bottom = 0.0
            for col, lab, colr in SEG:
                ax.bar(x0 + i, r[col], bottom=bottom, color=colr, width=.75, label=lab if (j == 0 and i == 0) or (col != "after_per100" and j == 0 and i == 1) else None)
                bottom += r[col]
            ax.text(x0 + i, 2.5, f"{r.after_per100:.0f}", ha="center", va="bottom", fontsize=7, color="white")
        ax.text(x0 + 1, 71, f"H < {t:g} s", ha="center", va="center", fontsize=8.5)
    ax.set_xticks([0, 1, 2, 4, 5, 6]); ax.set_xticklabels(xlab * 2, fontsize=7.5)
    ax.set_ylim(0, 75); ax.grid(axis="y", alpha=.2, lw=.5); ax.set_xlabel("BTW speed cap (km/h)", fontsize=8.5)
    ax.set_title(f"({tag}) {s}", loc="left", fontsize=8.5)
axes[0].set_ylabel("Events per 100 observed BTW events")
h, l = axes[0].get_legend_handles_labels(); seen = {}
for hh, ll in zip(h, l): seen.setdefault(ll, hh)
fig.tight_layout(rect=(0, 0.1, 1, 1))
fig.legend(seen.values(), seen.keys(), frameon=False, fontsize=7.2, ncol=2, loc="lower center", bbox_to_anchor=(0.5, -0.02)); save_fig(fig, "speed_cap_short_headway_exposure"); plt.show()

### Which leaders break up the following (case 2), by leader class

In [ ]:
bl = []
for s in SITES:
    g = d[(d.Site == s) & (d.Target_Class == "BTW")].reset_index(drop=True)
    for cap in CAPS:
        case = classify(g, cap)
        for c in LEADERS:
            m = (g.Leader_Class == c).to_numpy()
            bl.append({"Site": s, "Cap_kmh": cap, "Leader": c, "n": int(m.sum()), "share_case1": (case[m] == 1).mean(),
                       "share_case2": (case[m] == 2).mean(), "share_case3": (case[m] == 3).mean()})
bl = pd.DataFrame(bl); print(bl.round(3).to_string(index=False))

### Sensitivity: Shahjahanpur BTW, capped BTW leaders shorten the headway behind them

In [ ]:
g = d[(d.Site == "Shahjahanpur") & (d.Target_Class == "BTW")].reset_index(drop=True)
sens = []
for cap in CAPS:
    case, H0 = btw_scenario(g, cap); base = summarise(case, H0, g.Time_Headway_s.to_numpy())
    row = {"Cap_kmh": cap, "median_s_no_leader_effect": base["median_s"], "P(H<2)_no_leader_effect": base["P(H<2)"],
           "exp2_per100_no_leader_effect": base["exp2_after_per100"], "exp1.5_per100_no_leader_effect": base["exp1.5_after_per100"],
           "share_case3_behind_capped_BTW_leader": ((case == 3) & (g.Leader_Class == "BTW").to_numpy() & (g.Leading_Speed_kmh.to_numpy() > cap)).mean()}
    for k in ("est", "lo", "hi"):
        c1, H1 = btw_scenario(g, cap, leader_tr=TR_LS[k]); s1 = summarise(c1, H1, g.Time_Headway_s.to_numpy())
        row[f"median_s_TR_{k}"], row[f"P(H<2)_TR_{k}"] = s1["median_s"], s1["P(H<2)"]
        row[f"exp2_per100_TR_{k}"], row[f"exp1.5_per100_TR_{k}"] = s1["exp2_after_per100"], s1["exp1.5_after_per100"]
    sens.append(row)
sens = pd.DataFrame(sens); print(sens.round(3).to_string(index=False))

### Figure: what the cap does to the observed BTW following events

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 2.6))
labels, y = [], 0
for s in SITES:
    for cap in [25, 15]:
        r = scen[(scen.Site == s) & (scen.Cap_kmh == cap)].iloc[0]; left = 0.0
        for k in (1, 3, 2):
            w = 100 * r[f"share_case{k}"]
            ax.barh(y, w, left=left, color=CASE_COL[k], height=.62, label=CASES[k] if y == 0 else None, edgecolor="white", lw=.5)
            if w >= 6:
                ax.text(left + w / 2, y, f"{w:.0f}%", ha="center", va="center", fontsize=7.5, color="white" if k != 1 else PAL["ink"])
            left += w
        labels.append(f"{s}, {cap} km/h"); y += 1
    y += 0.4
ys = [0, 1, 2.4, 3.4]
ax.set_yticks(ys); ax.set_yticklabels(labels); ax.invert_yaxis()
ax.set_xlim(0, 100); ax.set_xlabel("Share of observed BTW following events (%)")
ax.legend(frameon=False, fontsize=7.5, ncol=3, loc="upper center", bbox_to_anchor=(0.45, -0.28))
for sp in ("top", "right"): ax.spines[sp].set_visible(False)
fig.tight_layout(); save_fig(fig, "speed_cap_outcome_shares"); plt.show()

### Figure: headways of the BTW events that keep following, against observed BTW and PR

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.2), sharey=True)
for ax, s, tag in zip(axes, SITES, "ab"):
    g = d[(d.Site == s) & (d.Target_Class == "BTW")].reset_index(drop=True)
    p = np.sort(d[(d.Site == s) & (d.Target_Class == "PR")].Time_Headway_s.to_numpy())
    curves = [("observed", g.Time_Headway_s.to_numpy(), "BTW observed")]
    for cap in [25, 15]:
        case, Hn = btw_scenario(g, cap); h = Hn[np.isfinite(Hn) & (Hn <= UPPER)]
        curves.append((cap, h, f"BTW still following, cap {cap} km/h (n={len(h)})"))
    for key, h, lab in curves:
        x = np.sort(h); ax.step(x, np.arange(1, len(x) + 1) / len(x), where="post", color=CAP_COL[key], lw=1.5, label=lab)
    ax.step(p, np.arange(1, len(p) + 1) / len(p), where="post", color=PAL["target"]["PR"], lw=1.4, ls="--", label="PR observed")
    ax.axvline(2, color="0.6", lw=.6, ls=":")
    ax.set_xlim(0.4, 5.1); ax.grid(alpha=.2, lw=.5); ax.set_xlabel("Time headway (s)")
    ax.set_title(f"({tag}) {s}", loc="left", fontsize=8.5); ax.legend(frameon=False, fontsize=6.6, loc="lower right")
axes[0].set_ylabel("Cumulative share")
fig.tight_layout(); save_fig(fig, "speed_cap_headway_ecdf"); plt.show()

### Export

In [ ]:
out = TAB / "05_speed_cap_scenarios.xlsx"
with pd.ExcelWriter(out) as xw:
    scen.assign(Cap_kmh=scen.Cap_kmh.astype(str)).to_excel(xw, sheet_name="scenarios", index=False)
    expo.assign(Cap_kmh=expo.Cap_kmh.astype(str)).to_excel(xw, sheet_name="short_headway_exposure", index=False)
    bl.to_excel(xw, sheet_name="cases_by_leader", index=False)
    sens.to_excel(xw, sheet_name="SP_BTW_leader_speed_sens", index=False)
    speed.to_excel(xw, sheet_name="observed_speeds", index=False)
    pd.DataFrame({"Parameter": ["policy", "caps_kmh", "cap_applies_to", "case_1", "case_2", "case_3", "following_window_upper_s",
                                "leader_speed_TR_SP_BTW", "bootstrap", "B", "SEED", "data04_sha256"],
                  "Value": ["S.R.O. No. 384-Law/2026, Bangladesh Gazette (extraordinary), 29 Oct 2026: rule 2(gha) 25 km/h design limit; rule 8(4) 15 km/h near schools, hospitals, places of worship and crowded places",
                            json.dumps(list(CAPS)), "BTW (e-rickshaw) only; PR not capped", "v <= cap: unchanged",
                            "v > cap and non-BTW leader with v_L > cap: following breaks up", "v > cap and (BTW leader or v_L <= cap): H' = H v / cap (spacing constant)",
                            UPPER, json.dumps({k: round(v, 4) for k, v in TR_LS.items()}), "events resampled within site x target x leader",
                            B, SEED, hashlib.sha256(DATA.read_bytes()).hexdigest()]}).to_excel(xw, sheet_name="settings", index=False)
cambria_xlsx(out); print("wrote", out.name)

## Step 6 — Dedicated e-rickshaw (BTW) lane scenario (by site)

**Scenario:** a lane reserved for e-rickshaws. Only BTWs use it, so:
- every leader is a BTW;
- no PR or other vehicle shares the lane;
- the general e-rickshaw limit of **25 km/h** applies (rule 2(ঘ), S.R.O. No. 384-Law/2026);
- following is centred, because there is no mixed traffic to stagger against.

**Basis in the data:** the observed BTW-follows-BTW events stand in for following inside such a lane. Two versions are reported side by side:

| Version | Events used | n (Shahjahanpur / Tikatuli) | Role |
|---|---|---|---|
| A. Dedicated lane | all BTW-led BTW events | 58 / 69 | main result; centring is part of the lane design and is not modelled |
| B. Dedicated lane, centred only | BTW-led BTW events that were centred | 10 / 37 | check; matches the scenario exactly, but Shahjahanpur rests on 10 events |

No off-centring effect is estimated or applied in either version.

**Cap:** both the follower and its BTW leader are held to 25 km/h, so following never breaks up. A follower faster than the cap runs at $v' = 25$ km/h with the same spacing, $H' = H\,v/25$ (as in step 5). An event pushed above 5 s leaves the following window.

**Compared with mixed traffic:** all observed BTW following events at the site (no cap), and mixed traffic under the same 25 km/h cap (step 5 logic: an event behind a faster uncapped leader breaks up and leaves the denominator).

**Reported:** median headway, and exposure to short headways ($H < 2$ s and $H < 1.5$ s) as the share of following events. For the lane every event keeps following, so this share is also exposure per 100 lane events.

**Leader-speed sensitivity (Shahjahanpur):** BTW followers there keep longer headways behind faster leaders (leader-adjusted time ratio 1.089 per 5 km/h, step 3). Capping BTW leaders at 25 km/h would therefore shorten the headways behind them; this is shown as a sensitivity check.

**Uncertainty:** 95% percentile intervals from a bootstrap that resamples BTW events within site × leader-class cells (B = 2,000) and takes the lane subset from each replicate, so lane and mixed traffic are compared on the same resample.

**Caution:** behaviour observed behind BTW leaders in mixed traffic is used as a proxy for behaviour inside a dedicated lane. A real lane may change speeds and spacing in ways these data cannot show.

**Outputs:** `outputs/tables/06_dedicated_btw_lane.xlsx`, and figures `dedicated_lane_exposure.png`, `dedicated_lane_headway_ecdf.png`.

In [ ]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path.cwd()))
from style import PAL, use_style
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from openpyxl import load_workbook
from openpyxl.styles import Font

HERE = Path.cwd()
DATA = HERE.parent / "outputs" / "data" / "data04.xlsx"
TAB = HERE.parent / "outputs" / "tables"; TAB.mkdir(parents=True, exist_ok=True)
FIG = HERE.parent / "outputs" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
FONT = "Cambria"
plt.rcParams.update({"font.family": FONT, "mathtext.fontset": "custom", "mathtext.rm": FONT,
                     "mathtext.it": FONT + ":italic", "mathtext.bf": FONT + ":bold", "font.size": 9})
def save_fig(fig, name):
    p = FIG / f"{name}.png"; p.unlink(missing_ok=True); fig.savefig(p, dpi=600, bbox_inches="tight"); print("saved", p.name)
def cambria_xlsx(path):
    wb = load_workbook(path)
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for c in row:
                c.font = Font(name=FONT, bold=(c.row == 1), size=11)
    wb.save(path)

### Settings

In [ ]:
SITES = ["Shahjahanpur", "Tikatuli"]
CAP = 25                           # km/h, general e-rickshaw limit (rule 2(gha))
LEADERS = ["4W", "BTW", "CNG", "MT_2W", "NMT"]
THRESH = [2.0, 1.5]
UPPER = 5.0
B = 2000; SEED = 20261107
SCEN = {"mixed_observed": "Mixed traffic, observed", "mixed_cap25": "Mixed traffic, 25 km/h cap",
        "lane_A": "Dedicated lane, 25 km/h (BTW-led)", "lane_B": "Dedicated lane, 25 km/h (BTW-led, centred only)"}
SCOL = PAL["lane"]
_tr = pd.read_excel(TAB / "03_covariate_time_ratios.xlsx", sheet_name="time_ratios")
_row = _tr[(_tr.Site == "Shahjahanpur") & (_tr.Class == "BTW") & (_tr.Model == "leader_adjusted") & (_tr.Variable == "Leader_speed_5kmh")].iloc[0]
TR_LS = {"est": float(_row.TR), "lo": float(_row.TR_lo95), "hi": float(_row.TR_hi95)}
print("Shahjahanpur BTW leader-speed TR per 5 km/h:", {k: round(v, 4) for k, v in TR_LS.items()})

### Data

In [ ]:
d = pd.read_excel(DATA, sheet_name="data04")
print("data04:", d.shape, " sha256:", hashlib.sha256(DATA.read_bytes()).hexdigest()[:16])
btw = {s: d[(d.Site == s) & (d.Target_Class == "BTW")].reset_index(drop=True) for s in SITES}
counts = []
for s in SITES:
    g = btw[s]; lead = g[g.Leader_Class == "BTW"]
    counts.append({"Site": s, "n_BTW_all": len(g), "n_BTW_led": len(lead), "n_BTW_led_centred": int((~lead.Off_Centered).sum()),
                   "share_follower_above_cap": (lead.Target_Speed_kmh > CAP).mean(), "share_leader_above_cap": (lead.Leading_Speed_kmh > CAP).mean(),
                   "mean_follower_speed_kmh": lead.Target_Speed_kmh.mean(), "mean_leader_speed_kmh": lead.Leading_Speed_kmh.mean()})
counts = pd.DataFrame(counts); print(counts.round(3).to_string(index=False))

### Scenario engine

In [ ]:
def capped(g, leader_tr=None):
    # both follower and BTW leader capped; spacing constant; returns new headways (NaN where following breaks up)
    H, v, vl = g.Time_Headway_s.to_numpy(float), g.Target_Speed_kmh.to_numpy(), g.Leading_Speed_kmh.to_numpy()
    lead_btw = (g.Leader_Class == "BTW").to_numpy()
    Hn = np.where(v > CAP, H * v / CAP, H)
    Hn = np.where((v > CAP) & ~lead_btw & (vl > CAP), np.nan, Hn)          # mixed traffic: faster uncapped leader, following breaks up
    if leader_tr is not None:
        lb = lead_btw & (vl > CAP)
        Hn = np.where(lb, Hn * leader_tr ** ((CAP - vl) / 5.0), Hn)
    return Hn

def summ(h):
    h = h[np.isfinite(h) & (h <= UPPER)]
    out = {"n_following": len(h), "median_s": np.median(h), "mean_s": h.mean()}
    for t in THRESH:
        out[f"share_H<{t:g}"] = (h < t).mean()
    return out

def scenarios(g, leader_tr=None):
    lead = (g.Leader_Class == "BTW").to_numpy(); cen = ~g.Off_Centered.to_numpy().astype(bool)
    Hc = capped(g, leader_tr)
    return {"mixed_observed": summ(g.Time_Headway_s.to_numpy(float)), "mixed_cap25": summ(Hc),
            "lane_A": summ(Hc[lead]), "lane_B": summ(Hc[lead & cen])}

def strat_resample(g, rng):
    parts = [g[g.Leader_Class == c] for c in LEADERS]
    return pd.concat([q.iloc[rng.integers(0, len(q), len(q))] for q in parts], ignore_index=True)

### Results with bootstrap intervals

In [ ]:
rng = np.random.default_rng(SEED)
rows = []
METRICS = ["median_s", "mean_s"] + [f"share_H<{t:g}" for t in THRESH]
for s in SITES:
    g = btw[s]; est = scenarios(g)
    boot = {k: [] for k in SCEN}; dboot = {k: [] for k in SCEN}
    for _ in range(B):
        sb = scenarios(strat_resample(g, rng))
        for k in SCEN:
            boot[k].append(sb[k]); dboot[k].append({m: sb[k][m] - sb["mixed_observed"][m] for m in METRICS})
    for k, lab in SCEN.items():
        bb, db = pd.DataFrame(boot[k]), pd.DataFrame(dboot[k])
        r = {"Site": s, "Scenario": lab, "key": k, **est[k]}
        for m in METRICS:
            r[m + "_lo95"], r[m + "_hi95"] = np.percentile(bb[m], [2.5, 97.5])
            r["d_" + m] = est[k][m] - est["mixed_observed"][m]
            r["d_" + m + "_lo95"], r["d_" + m + "_hi95"] = np.percentile(db[m], [2.5, 97.5])
        rows.append(r)
res = pd.DataFrame(rows)
show = ["Site", "Scenario", "n_following", "median_s", "median_s_lo95", "median_s_hi95", "share_H<2", "share_H<2_lo95", "share_H<2_hi95",
        "d_share_H<2", "d_share_H<2_lo95", "d_share_H<2_hi95", "share_H<1.5", "share_H<1.5_lo95", "share_H<1.5_hi95",
        "d_share_H<1.5", "d_share_H<1.5_lo95", "d_share_H<1.5_hi95"]
print(res[show].round(3).to_string(index=False))

### Sensitivity: Shahjahanpur, capped BTW leaders shorten the headways behind them

In [ ]:
g = btw["Shahjahanpur"]; sens = []
for k in ("est", "lo", "hi"):
    sc = scenarios(g, leader_tr=TR_LS[k])
    for key in ("lane_A", "lane_B"):
        sens.append({"Site": "Shahjahanpur", "TR_leader_speed": k, "TR_value": TR_LS[k], "Scenario": SCEN[key], **sc[key]})
base = scenarios(g)
for key in ("lane_A", "lane_B"):
    sens.append({"Site": "Shahjahanpur", "TR_leader_speed": "none", "TR_value": 1.0, "Scenario": SCEN[key], **base[key]})
sens = pd.DataFrame(sens); print(sens.round(3).to_string(index=False))

### Figure: exposure to short headways, mixed traffic vs dedicated lane

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.3), sharey=True)
keys = list(SCEN); w = 0.2
for ax, s, tag in zip(axes, SITES, "ab"):
    sub = res[res.Site == s].set_index("key")
    for i, k in enumerate(keys):
        x = np.arange(len(THRESH)) + (i - 1.5) * w
        y = [100 * sub.loc[k, f"share_H<{t:g}"] for t in THRESH]
        lo = [100 * sub.loc[k, f"share_H<{t:g}_lo95"] for t in THRESH]; hi = [100 * sub.loc[k, f"share_H<{t:g}_hi95"] for t in THRESH]
        ax.bar(x, y, width=w, color=SCOL[k], label=f"{SCEN[k]}" if tag == "a" else None, edgecolor="white", lw=.4)
        ax.errorbar(x, y, yerr=[np.array(y) - lo, np.array(hi) - y], fmt="none", ecolor="k", elinewidth=.8, capsize=1.8)
        for xx, yy in zip(x, y):
            ax.text(xx, 2, f"{yy:.0f}", ha="center", va="bottom", fontsize=6.5, color="white")
    ax.set_xticks(range(len(THRESH))); ax.set_xticklabels([f"H < {t:g} s" for t in THRESH])
    ax.set_title(f"({tag}) {s}  (lane n = {int(sub.loc['lane_A', 'n_following'])}; centred n = {int(sub.loc['lane_B', 'n_following'])})",
                 loc="left", fontsize=8.5)
    ax.set_ylim(0, 100); ax.grid(axis="y", alpha=.2, lw=.5)
axes[0].set_ylabel("Share of following events (%)")
fig.tight_layout(rect=(0, 0.1, 1, 1))
fig.legend(frameon=False, fontsize=7.2, ncol=2, loc="lower center", bbox_to_anchor=(0.5, -0.02))
save_fig(fig, "dedicated_lane_exposure"); plt.show()

### Figure: headway distribution, mixed traffic vs dedicated lane

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.2), sharey=True)
for ax, s, tag in zip(axes, SITES, "ab"):
    g = btw[s]; lead = (g.Leader_Class == "BTW").to_numpy(); cen = ~g.Off_Centered.to_numpy().astype(bool); Hc = capped(g)
    curves = {"mixed_observed": g.Time_Headway_s.to_numpy(float), "lane_A": Hc[lead], "lane_B": Hc[lead & cen]}
    for k, h in curves.items():
        x = np.sort(h[np.isfinite(h) & (h <= UPPER)])
        ax.step(x, np.arange(1, len(x) + 1) / len(x), where="post", color=SCOL[k],
                lw=1.5, ls="-" if k != "lane_B" else "--", label=f"{SCEN[k]} (n={len(x)})")
    ax.axvline(2, color="0.6", lw=.6, ls=":"); ax.axvline(1.5, color="0.6", lw=.6, ls=":")
    ax.set_xlim(0.4, 5.1); ax.grid(alpha=.2, lw=.5); ax.set_xlabel("BTW follower time headway (s)")
    ax.set_title(f"({tag}) {s}", loc="left", fontsize=8.5); ax.legend(frameon=False, fontsize=6.6, loc="lower right")
axes[0].set_ylabel("Cumulative share")
fig.tight_layout(); save_fig(fig, "dedicated_lane_headway_ecdf"); plt.show()

### Export

In [ ]:
out = TAB / "06_dedicated_btw_lane.xlsx"
with pd.ExcelWriter(out) as xw:
    res.drop(columns="key").to_excel(xw, sheet_name="scenarios", index=False)
    counts.to_excel(xw, sheet_name="lane_events", index=False)
    sens.to_excel(xw, sheet_name="SP_leader_speed_sens", index=False)
    pd.DataFrame({"Parameter": ["policy", "cap_kmh", "lane_A", "lane_B", "headway_under_cap", "mixed_cap25", "thresholds_s",
                                "following_window_upper_s", "leader_speed_TR_SP_BTW", "bootstrap", "B", "SEED", "data04_sha256"],
                  "Value": ["S.R.O. No. 384-Law/2026, rule 2(gha): 25 km/h e-rickshaw limit", CAP,
                            "BTW followers behind BTW leaders, all lateral positions (main)", "BTW followers behind BTW leaders, centred only (check)",
                            "H' = H v / 25 for v > 25 (spacing constant); leader also capped, so no break-up in the lane",
                            "notebook 05 logic: v > 25 behind a faster uncapped leader breaks up", json.dumps(THRESH), UPPER,
                            json.dumps({k: round(v, 4) for k, v in TR_LS.items()}), "BTW events resampled within site x leader; lane subset per replicate",
                            B, SEED, hashlib.sha256(DATA.read_bytes()).hexdigest()]}).to_excel(xw, sheet_name="settings", index=False)
cambria_xlsx(out); print("wrote", out.name)

## Step 7 — Sensitivity to the 0.5–5 s extraction window

Events were extracted only when $0.5 \le H \le 5$ s, so the window can be **narrowed** but not widened. The key results are refitted on narrower windows, each with the likelihood truncated to its own bounds. This tests whether they depend on where the extraction rule draws the lines:
- trimming the shortest headways (the end most exposed to coding error);
- trimming the longest headways (the end closest to free flow).

| Window (s) | Removes |
|---|---|
| 0.5–5.0 | nothing (main analysis) |
| 0.75–5.0, 1.0–5.0 | the shortest headways |
| 0.5–4.5, 0.5–4.0 | the longest headways |
| 1.0–4.0 | both ends |

**Re-estimated for each window:**
1. Best distribution by AIC among the five candidates (step 1), and whether Weibull stays within 2 AIC.
2. Leader time ratios of the truncated Weibull AFT (step 2, reference 4W), 95% Wald intervals.
3. Share of the PR–BTW log-headway contrast due to speed (step 4), with a stratified bootstrap interval (B = 1,000).

Where a narrow window puts its upper bound close to the PR mode (about 3 s) and a leader cell has only about ten events, the leader-conditioned Weibull fit for PR followers is not identified. Such time ratios are marked as missing, not reported.

**Outputs:** `outputs/tables/07_window_sensitivity.xlsx`, and figure `window_sensitivity.png`.

In [ ]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path.cwd()))
from style import PAL, use_style
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from openpyxl import load_workbook
from openpyxl.styles import Font
import hwdist as hd

HERE = Path.cwd()
DATA = HERE.parent / "outputs" / "data" / "data04.xlsx"
TAB = HERE.parent / "outputs" / "tables"; TAB.mkdir(parents=True, exist_ok=True)
FIG = HERE.parent / "outputs" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
FONT = "Cambria"
plt.rcParams.update({"font.family": FONT, "mathtext.fontset": "custom", "mathtext.rm": FONT,
                     "mathtext.it": FONT + ":italic", "mathtext.bf": FONT + ":bold", "font.size": 9})
def save_fig(fig, name):
    p = FIG / f"{name}.png"; p.unlink(missing_ok=True); fig.savefig(p, dpi=600, bbox_inches="tight"); print("saved", p.name)
def cambria_xlsx(path):
    wb = load_workbook(path)
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for c in row:
                c.font = Font(name=FONT, bold=(c.row == 1), size=11)
    wb.save(path)

### Settings

In [ ]:
WINDOWS = [(0.5, 5.0), (0.75, 5.0), (1.0, 5.0), (0.5, 4.5), (0.5, 4.0), (1.0, 4.0)]
WLAB = {w: f"{w[0]:g}–{w[1]:g} s" for w in WINDOWS}
CASES = [("Shahjahanpur", "PR"), ("Shahjahanpur", "BTW"), ("Tikatuli", "PR"), ("Tikatuli", "BTW")]
SITES = ["Shahjahanpur", "Tikatuli"]
LEADERS = ["4W", "BTW", "CNG", "MT_2W", "NMT"]; NONREF = LEADERS[1:]
CANDIDATES = ["lognormal", "loglogistic", "weibull", "gamma", "gengamma"]
B = 1000; SEED = 20261107

### Data

In [ ]:
d = pd.read_excel(DATA, sheet_name="data04")
print("data04:", d.shape, " sha256:", hashlib.sha256(DATA.read_bytes()).hexdigest()[:16])
d["logH"] = np.log(d.Time_Headway_s); d["logv"] = np.log(d.Target_Speed_kmh / 3.6); d["logS"] = d.logH + d.logv
cnt = pd.DataFrame([{"Window": WLAB[w], **{f"{s} {t}": int(((d.Site == s) & (d.Target_Class == t) & d.Time_Headway_s.between(*w)).sum())
                                         for s, t in CASES}} for w in WINDOWS])
print(cnt.to_string(index=False))

### 1–2. Best distribution and Weibull leader time ratios in each window

In [ ]:
fam_rows, tr_rows = [], []
for w in WINDOWS:
    lo, hi = w
    for c in CASES:
        g = d[(d.Site == c[0]) & (d.Target_Class == c[1]) & d.Time_Headway_s.between(lo, hi)]
        x = g.Time_Headway_s.to_numpy(float)
        aic = {f: hd.fit(x, f, lower=lo, upper=hi)["AIC"] for f in CANDIDATES}
        best = min(aic, key=aic.get)
        fam_rows.append({"Window": WLAB[w], "Site": c[0], "Class": c[1], "n": len(x), "best_family": hd.LABELS[best],
                         "dAIC_weibull": aic["weibull"] - aic[best], "weibull_within_2": aic["weibull"] - aic[best] < 2})
        X = np.column_stack([np.ones(len(g))] + [(g.Leader_Class == L).to_numpy(float) for L in NONREF])
        r = hd.fit(x, "weibull", X=X, lower=lo, upper=hi); V = hd.covariance(r, x)
        for j, L in enumerate(NONREF, start=1):
            se = float(np.sqrt(max(V[j, j], 0))); b = float(r["beta"][j])
            tr_rows.append({"Window": WLAB[w], "Site": c[0], "Class": c[1], "Leader": L, "n_leader": int((g.Leader_Class == L).sum()),
                            "TR": np.exp(b), "TR_lo95": np.exp(b - 1.96 * se), "TR_hi95": np.exp(b + 1.96 * se)})
fam = pd.DataFrame(fam_rows); trw = pd.DataFrame(tr_rows)
# a fit is "not identified" when its interval is not finite or the ratio is implausible (narrow window near the PR mode, small leader cells)
trw["identified"] = np.isfinite(trw.TR_lo95) & np.isfinite(trw.TR_hi95) & trw.TR.between(0.1, 10) & (trw.TR_hi95 / trw.TR_lo95 < 100)
for col in ["TR", "TR_lo95", "TR_hi95"]:
    trw.loc[~trw.identified, col] = np.nan
print("leader time ratios not identified:")
print(trw[~trw.identified][["Window", "Site", "Class", "Leader", "n_leader"]].to_string(index=False))
print(fam.pivot_table(index="Window", columns=["Site", "Class"], values="best_family", aggfunc="first", sort=False).to_string())
print(fam.pivot_table(index="Window", columns=["Site", "Class"], values="dAIC_weibull", sort=False).round(2).to_string())

### 3. Speed share of the PR–BTW contrast in each window

In [ ]:
def speed_share(g, w_shares):
    m = g.groupby(["Target_Class", "Leader_Class"])[["logH", "logv"]].mean()
    dH = sum(w_shares[L] * (m.loc[("PR", L), "logH"] - m.loc[("BTW", L), "logH"]) for L in LEADERS)
    dv = sum(w_shares[L] * (m.loc[("PR", L), "logv"] - m.loc[("BTW", L), "logv"]) for L in LEADERS)
    return dH, -dv / dH

rng = np.random.default_rng(SEED); dec_rows = []
for w in WINDOWS:
    for s in SITES:
        g = d[(d.Site == s) & d.Time_Headway_s.between(*w)].reset_index(drop=True)
        ws = g.Leader_Class.value_counts(normalize=True).reindex(LEADERS)
        dH, sh = speed_share(g, ws)
        cells = [np.flatnonzero(((g.Target_Class == t) & (g.Leader_Class == L)).to_numpy()) for t in ["PR", "BTW"] for L in LEADERS]
        bs = []
        for _ in range(B):
            idx = np.concatenate([rng.choice(ix, size=len(ix), replace=True) for ix in cells])
            bs.append(speed_share(g.iloc[idx], ws)[1])
        dec_rows.append({"Window": WLAB[w], "Site": s, "n": len(g), "headway_ratio_PR_BTW": np.exp(dH), "speed_share": sh,
                         "speed_share_lo95": np.percentile(bs, 2.5), "speed_share_hi95": np.percentile(bs, 97.5)})
dec = pd.DataFrame(dec_rows); print(dec.round(3).to_string(index=False))

### Figure: leader time ratios and speed share across windows

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(7.4, 3.0), gridspec_kw={"width_ratios": [1, 1, 0.9]})
yl = np.arange(len(WINDOWS))[::-1]
for ax, c, tag in zip(axes[:2], [("Shahjahanpur", "BTW"), ("Tikatuli", "BTW")], "ab"):
    off = dict(zip(NONREF, np.linspace(0.27, -0.27, len(NONREF))))
    for L in NONREF:
        t = trw[(trw.Site == c[0]) & (trw.Class == c[1]) & (trw.Leader == L)].set_index("Window").loc[[WLAB[w] for w in WINDOWS]]
        y = yl + off[L]
        ax.errorbar(t.TR, y, xerr=[t.TR - t.TR_lo95, t.TR_hi95 - t.TR], fmt="o", ms=3, color=PAL["leader"][L], elinewidth=1, capsize=0,
                    label=L)
    ax.axvline(1, color="0.4", lw=.7); ax.set_xscale("log"); ax.set_xlim(0.4, 1.6); ax.set_xticks([0.5, 0.7, 1, 1.4]); ax.set_xticklabels(["0.5", "0.7", "1", "1.4"])
    ax.minorticks_off(); ax.grid(axis="x", alpha=.25, lw=.5); ax.set_xlabel("Time ratio vs 4W leader")
    ax.set_title(f"({tag}) {c[0]}, BTW followers", loc="left", fontsize=8.5)
    for k in range(len(WINDOWS) - 1): ax.axhline(k + 0.5, color="0.9", lw=.6)
axes[0].set_yticks(yl); axes[0].set_yticklabels([WLAB[w] for w in WINDOWS]); axes[1].set_yticks(yl); axes[1].set_yticklabels([])
axes[0].set_ylabel("Extraction window")
ax = axes[2]
for s, dy, mk in [("Shahjahanpur", 0.12, "o"), ("Tikatuli", -0.12, "s")]:
    t = dec[dec.Site == s].set_index("Window").loc[[WLAB[w] for w in WINDOWS]]
    ax.errorbar(100 * t.speed_share, yl + dy, xerr=[100 * (t.speed_share - t.speed_share_lo95), 100 * (t.speed_share_hi95 - t.speed_share)],
                fmt=mk, ms=3.5, color=PAL["ink"] if s == "Shahjahanpur" else PAL["muted"], mfc=PAL["ink"] if s == "Shahjahanpur" else "white", elinewidth=1, capsize=0, label=s)
ax.set_yticks(yl); ax.set_yticklabels([]); ax.axvline(100, color="0.6", lw=.6, ls=":"); ax.set_xlim(0, 220); ax.set_xticks([0, 50, 100, 150, 200]); ax.grid(axis="x", alpha=.25, lw=.5)
ax.set_xlabel("Speed share of PR–BTW contrast (%)"); ax.set_title("(c) Spacing–speed split", loc="left", fontsize=8.5)
for k in range(len(WINDOWS) - 1): ax.axhline(k + 0.5, color="0.9", lw=.6)
h1, l1 = axes[0].get_legend_handles_labels(); h3, l3 = ax.get_legend_handles_labels()
fig.tight_layout(rect=(0, 0.1, 1, 1))
fig.legend(h1 + h3, l1 + l3, frameon=False, fontsize=7.2, ncol=6, loc="lower center", bbox_to_anchor=(0.5, -0.01))
save_fig(fig, "window_sensitivity"); plt.show()

### Export

In [ ]:
out = TAB / "07_window_sensitivity.xlsx"
with pd.ExcelWriter(out) as xw:
    cnt.to_excel(xw, sheet_name="n_by_window", index=False)
    fam.to_excel(xw, sheet_name="best_family", index=False)
    trw.to_excel(xw, sheet_name="weibull_leader_TR", index=False)
    dec.to_excel(xw, sheet_name="speed_share", index=False)
    pd.DataFrame({"Parameter": ["windows_s", "B", "SEED", "data04_sha256"],
                  "Value": [json.dumps(WINDOWS), B, SEED, hashlib.sha256(DATA.read_bytes()).hexdigest()]}).to_excel(xw, sheet_name="settings", index=False)
cambria_xlsx(out); print("wrote", out.name)

## Manuscript tables — Manuscript and appendix tables

Collects the results of steps 1–7 into one workbook, `outputs/Manuscript_Tables.xlsx`, with one table per sheet. Each table keeps only the columns needed in the manuscript or appendix. The full result tables stay in `outputs/tables/`.

**Style**, following the TRB manuscript:
- estimates with 95% intervals in brackets;
- p-values to three decimals ("<0.001" below that);
- goodness-of-fit results marked (A) accepted or (R) rejected at 5%;
- percentages to one decimal;
- negative numbers with a true minus sign.

Each sheet has a title row, a header row, the table, and a notes row.

**Run order:** notebook 1, then 2, then 3.

In [ ]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path.cwd()))
from style import PAL, use_style
from pathlib import Path
import numpy as np
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

HERE = Path.cwd()
DATA = HERE.parent / "outputs" / "data" / "data04.xlsx"
TAB = HERE.parent / "outputs" / "tables"
FIG = HERE.parent / "outputs" / "figures"
OUT = HERE.parent / "outputs" / "Manuscript_Tables.xlsx"
need = ["01_truncated_distribution_fit", "02_leader_conditioned_aft", "02b_leader_distribution_shifts", "03_covariate_time_ratios",
        "04_spacing_speed_decomposition", "05_speed_cap_scenarios", "06_dedicated_btw_lane", "07_window_sensitivity"]
missing = [n for n in need if not (TAB / f"{n}.xlsx").exists()]
assert not missing, f"run these notebooks first: {missing}"
R = {n: pd.read_excel(TAB / f"{n}.xlsx", sheet_name=None) for n in need}
d = pd.read_excel(DATA, sheet_name="data04")
CASES = [("Shahjahanpur", "PR"), ("Shahjahanpur", "BTW"), ("Tikatuli", "PR"), ("Tikatuli", "BTW")]
CASE_LAB = {c: f"{c[0]}, {c[1]}" for c in CASES}
LEADERS = ["4W", "BTW", "CNG", "MT_2W", "NMT"]
FAM_ORDER = ["Lognormal", "Log-logistic", "Weibull", "Gamma", "Generalized gamma"]
print("inputs found:", len(R), "result workbooks")

### Formatting helpers

In [ ]:
MINUS = "−"
def num(x, d=3):
    if x is None or (isinstance(x, float) and np.isnan(x)): return "—"
    s = f"{x:.{d}f}"
    return s.replace("-", MINUS) if s.startswith("-") and float(s) != 0 else s.lstrip("-")
def ci(e, lo, hi, d=3):
    return f"{num(e, d)} ({num(lo, d)} to {num(hi, d)})"
def pct(x, d=1): return num(100 * x, d)
def pct_ci(e, lo, hi, d=1): return ci(100 * e, 100 * lo, 100 * hi, d)
def signed(x, d=1):
    s = num(x, d); return s if s.startswith(MINUS) or s in ("0.0", "0", "—") else "+" + s
def signed_ci(e, lo, hi, d=1): return f"{signed(e, d)} ({signed(lo, d)} to {signed(hi, d)})"
def pval(p):
    if p is None or (isinstance(p, float) and np.isnan(p)): return "—"
    return "<0.001" if p < 0.001 else f"{p:.3f}"
def gof(p): return f"{pval(p)} ({'A' if p >= 0.05 else 'R'})"
def n_pct(k, n): return f"{k} ({100 * k / n:.1f}%)"

TABLES = []        # (sheet, title, DataFrame, notes)
def add(sheet, title, df, notes):
    TABLES.append((sheet, title, df.reset_index(drop=True), notes))

### Table 1 — Sample description

In [ ]:
rows = []
def block(label, fn):
    for cat, f in fn:
        r = {"Variable": label, "Category": cat}
        for c in CASES:
            g = d[(d.Site == c[0]) & (d.Target_Class == c[1])]; r[CASE_LAB[c]] = f(g)
        rows.append(r); label = ""
block("Following events", [("n", lambda g: str(len(g)))])
block("Leader class", [(L, (lambda L: lambda g: n_pct(int((g.Leader_Class == L).sum()), len(g)))(L)) for L in LEADERS])
block("Lateral position", [("Off-centred", lambda g: n_pct(int(g.Off_Centered.sum()), len(g)))])
block("Occupancy", [("Occupied", lambda g: n_pct(int(g.Occupied.sum()), len(g)))])
ms = lambda col: (lambda g: f"{num(g[col].mean(), 2)} ± {num(g[col].std(), 2)}")
rg = lambda col: (lambda g: f"{num(g[col].min(), 2)} to {num(g[col].max(), 2)}")
block("Time headway (s)", [("Mean ± SD", ms("Time_Headway_s")), ("Median", lambda g: num(g.Time_Headway_s.median(), 2)),
                           ("Range", rg("Time_Headway_s"))])
block("Follower speed (km/h)", [("Mean ± SD", ms("Target_Speed_kmh"))])
block("Leader speed (km/h)", [("Mean ± SD", ms("Leading_Speed_kmh"))])
t1 = pd.DataFrame(rows)
add("T1_Sample", "Table 1. Following events by site and target class", t1,
    "Events extracted only when 0.5 ≤ H ≤ 5 s. Leader classes after pooling: 4W (car, SUV, utility, human hauler), BTW (BTW, modified BTW), "
    "CNG auto-rickshaw, MT_2W (motorcycle), NMT (PR, van, bicycle). Speeds are space-mean speeds over the segment.")
t1

### Table 2 — Truncated distribution fits (step 1)

In [ ]:
f = R["01_truncated_distribution_fit"]["fits_gof"]
f = f[f.Family.isin(FAM_ORDER)].copy()
rows = []
for c in CASES:
    g = f[(f.Site == c[0]) & (f.Class == c[1])].sort_values("AIC")
    for i, (_, r) in enumerate(g.iterrows()):
        rows.append({"Case": f"{CASE_LAB[c]} (n = {int(r.n)})" if i == 0 else "", "Distribution": r.Family, "AIC": num(r.AIC, 2),
                     "ΔAIC": num(r.dAIC, 2), "Akaike weight": num(r.Akaike_weight, 3),
                     "KS p": gof(r.KS_p), "AD p": gof(r.AD_p), "CvM p": gof(r.CvM_p)})
add("T2_Distribution_fit", "Table 2. Truncated headway distributions by site and target class", pd.DataFrame(rows),
    "Maximum likelihood truncated to the 0.5–5 s extraction window. p-values from a refitted parametric bootstrap (999 replicates) of the "
    "KS, AD and CvM statistics on the probability-integral transform; (A) not rejected, (R) rejected at 5%. Families ordered by AIC within each case.")
TABLES[-1][2].head(6)

### Table 3 — Leader-conditioned AFT: choice of family and fit (step 2)

In [ ]:
fc = R["02_leader_conditioned_aft"]["family_choice"].copy()
fl = R["02_leader_conditioned_aft"]["fits_LR_gof"]
cand = fl[fl.Family.isin(FAM_ORDER)]
adq = cand.groupby("Family").adequate_KS_AD_CvM.sum()
t3a = pd.DataFrame({"Distribution": fc.Family, "Σ AIC (4 cases)": fc.sum_AIC_leader.map(lambda x: num(x, 2)),
                    "ΔΣ AIC": fc.d_sum_AIC.map(lambda x: num(x, 2)),
                    "Joint LR vs generalized gamma (4 df), p": [("—" if np.isnan(a) else f"{num(a, 2)}, {pval(p)}") for a, p in zip(fc.joint_LR_vs_GG, fc.joint_LR_p)],
                    "Cases adequate (of 4)": [int(adq.get(fm, 0)) for fm in fc.Family]})
add("T3a_Family_choice", "Table 3a. Choice of one distribution for the leader-conditioned AFT model", t3a,
    "Leader-conditioned truncated AFT (location = intercept + four leader-class indicators, reference 4W). Chosen family: the best-ranked "
    "two-parameter family not rejected by the joint likelihood-ratio test against the generalized gamma (Weibull). Adequate = KS, AD and CvM all p ≥ 0.05.")
s = R["02_leader_conditioned_aft"]["summary"]
rows = []
for c in CASES:
    r = s[(s.Site == c[0]) & (s.Class == c[1])].iloc[0]
    rows.append({"Case": f"{CASE_LAB[c]} (n = {int(r.n)})", "AIC, no leader": num(r.AIC_uncond, 2), "AIC, leader class": num(r.AIC_leader, 2),
                 "ΔAIC": num(r.AIC_change, 2), "LR leader class (4 df), p": f"{num(r.LR_stat, 2)}, {pval(r.LR_p)}",
                 "KS p": gof(r.KS_p), "AD p": gof(r.AD_p), "CvM p": gof(r.CvM_p)})
add("T3b_Weibull_leader_model", "Table 3b. Leader-conditioned Weibull AFT by site and target class", pd.DataFrame(rows),
    "ΔAIC = AIC with leader class − AIC without. Goodness of fit from a refitted parametric bootstrap (999 replicates).")
TABLES[-1][2]

### Table 4 — Time ratios by leader class (step 2)

In [ ]:
tr = R["02_leader_conditioned_aft"]["time_ratios"]
rows = []
for L in ["BTW", "CNG", "MT_2W", "NMT"]:
    r = {"Leader class": L}
    for c in CASES:
        x = tr[(tr.Site == c[0]) & (tr.Class == c[1]) & (tr.Leader == L)].iloc[0]
        r[CASE_LAB[c]] = ci(x.TR, x.TR_lo95, x.TR_hi95) + (" *" if x.TR_hi95 < 1 or x.TR_lo95 > 1 else "")
    rows.append(r)
nref = {CASE_LAB[c]: str(int(tr[(tr.Site == c[0]) & (tr.Class == c[1])].n_ref.iloc[0])) for c in CASES}
rows.insert(0, {"Leader class": "4W (reference, n)", **nref})
add("T4_Leader_time_ratios", "Table 4. Time ratios of the headway distribution by leader class (Weibull AFT, reference 4W)", pd.DataFrame(rows),
    "Time ratio exp(β) with 95% Wald interval. A ratio below 1 compresses the whole headway distribution relative to following a four-wheeler. "
    "* interval excludes 1.")
TABLES[-1][2]

### Table 5 — Leader shifts against the main distribution (step 2b)

In [ ]:
lv = R["02b_leader_distribution_shifts"]["leader_vs_main"]
sp = R["02b_leader_distribution_shifts"]["type_shift_vs_spread"]
TYPE = {"compresses (shorter headways)": "compresses", "stretches (longer headways)": "stretches", "no clear shift": "—"}
rows = []
for L in LEADERS:
    r = {"Leader class": L}
    for c in CASES:
        x = lv[(lv.Site == c[0]) & (lv.Class == c[1]) & (lv.Leader == L)].iloc[0]
        r[CASE_LAB[c]] = f"{signed_ci(x.d_median_pct, 100 * (x.median_ratio_lo95 - 1), 100 * (x.median_ratio_hi95 - 1))} {TYPE[x.Effect_type]}".rstrip(" —")
    rows.append(r)
r = {"Leader class": "Shift-only vs shift and spread, LR (4 df) p"}
for c in CASES:
    x = sp[(sp.Site == c[0]) & (sp.Class == c[1])].iloc[0]; r[CASE_LAB[c]] = pval(x.LR_p)
rows.append(r)
add("T5_Leader_shifts", "Table 5. Change in median headway behind each leader class relative to the case's main distribution (%)", pd.DataFrame(rows),
    "Main distribution = mixture of the leader-specific Weibull fits weighted by the observed leader shares. 95% intervals from 4,000 draws of the "
    "parameter vector. 'compresses' / 'stretches' = interval wholly below / above 0. Last row: does the leader also change the spread (shape)?")
TABLES[-1][2]

### Table 6 — Other variables as time ratios (step 3)

In [ ]:
cv = R["03_covariate_time_ratios"]["crude_vs_adjusted"]
VAR = {"Off_Centered": "Off-centred following", "Occupied": "Occupied (passengers)", "Leader_speed_5kmh": "Leader speed (+5 km/h)"}
rows = []
for v, vl in VAR.items():
    for i, c in enumerate(CASES):
        x = cv[(cv.Site == c[0]) & (cv.Class == c[1]) & (cv.Variable == v)].iloc[0]
        rows.append({"Variable": vl if i == 0 else "", "Case": CASE_LAB[c],
                     "Crude TR (95% CI)": ci(x.crude_TR, x.crude_TR_lo95, x.crude_TR_hi95), "Crude Holm p": pval(x.crude_LR_p_Holm),
                     "Leader-adjusted TR (95% CI)": ci(x.leader_adjusted_TR, x.leader_adjusted_TR_lo95, x.leader_adjusted_TR_hi95),
                     "Leader-adjusted Holm p": pval(x.leader_adjusted_LR_p_Holm)})
add("T6_Covariate_time_ratios", "Table 6. Time ratios of off-centred following, occupancy and leader speed (truncated Weibull AFT)", pd.DataFrame(rows),
    "Each variable entered alone: crude = variable only; leader-adjusted = added to the leader-class model. Likelihood-ratio p-values, "
    "Holm-adjusted across the three variables within each case and model. Leader speed centred at the case mean.")
TABLES[-1][2].head(4)

### Table 7 — Spacing–speed decomposition (step 4)

In [ ]:
de = R["04_spacing_speed_decomposition"]["decomposition"]
bt = R["04_spacing_speed_decomposition"]["by_site_target"]
rows = []
for site in ["Shahjahanpur", "Tikatuli"]:
    gp = bt[(bt.Site == site) & (bt.Target_Class == "PR")].iloc[0]; gb = bt[(bt.Site == site) & (bt.Target_Class == "BTW")].iloc[0]
    q = de[de.Site == site].set_index("Component")
    spec = [("Headway (s)", gp.gm_headway_s, gb.gm_headway_s, "Headway", 2),
            ("Spacing (m)", gp.gm_spacing_m, gb.gm_spacing_m, "Spacing part", 2),
            ("Speed (km/h)", gp.gm_speed_kmh, gb.gm_speed_kmh, "Speed part", 1)]
    for i, (lab, vp, vb, comp, dd) in enumerate(spec):
        x = q.loc[comp]
        rows.append({"Site": site if i == 0 else "", "Quantity": lab, "PR": num(vp, dd), "BTW": num(vb, dd),
                     "Contrast (%, 95% CI)": signed_ci(x.pct, x.pct_lo95, x.pct_hi95),
                     "Share of log headway contrast (%)": "100" if comp == "Headway" else pct_ci(x.share_of_headway_contrast, x.share_lo95, x.share_hi95, 0)})
add("T7_Decomposition", "Table 7. PR–BTW headway contrast split into spacing and speed parts (same leader mix)", pd.DataFrame(rows),
    "Geometric means. log H = log S − log v; contrasts PR − BTW, standardised to the site's leader shares; headway % = PR ÷ BTW − 1; "
    "spacing % = PR ÷ BTW spacing − 1; speed % = BTW ÷ PR speed − 1. The spacing and speed parts multiply to the headway contrast. "
    "Stratified bootstrap (4,000 replicates).")
TABLES[-1][2]

### Table 8 — E-rickshaw speed limits (step 5)

In [ ]:
sc = R["05_speed_cap_scenarios"]["scenarios"]; ex = R["05_speed_cap_scenarios"]["short_headway_exposure"]
sc["Cap_kmh"] = sc.Cap_kmh.astype(str); ex["Cap_kmh"] = ex.Cap_kmh.astype(str)
LIM = {"observed": "No limit (observed)", "25": "25 km/h (general)", "15": "15 km/h (zone)"}
rows = []
for site in ["Shahjahanpur", "Tikatuli"]:
    for i, cap in enumerate(["observed", "25", "15"]):
        x = sc[(sc.Site == site) & (sc.Cap_kmh == cap)].iloc[0]
        e2 = ex[(ex.Site == site) & (ex.Cap_kmh == cap) & (ex.Threshold_s == 2.0)].iloc[0]
        e15 = ex[(ex.Site == site) & (ex.Cap_kmh == cap) & (ex.Threshold_s == 1.5)].iloc[0]
        obs = cap == "observed"
        rows.append({"Site": site if i == 0 else "", "Speed limit": LIM[cap],
                     "Not affected (%)": pct(x.share_case1), "Follows at the limit (%)": pct(x.share_case3),
                     "Following breaks up (%)": "0.0" if obs else pct_ci(x.share_case2, x.share_case2_lo95, x.share_case2_hi95),
                     "Median, still following (s)": num(x.median_s, 2),
                     "Median change: selection / speed (%)": "—" if obs else f"{signed(x.d_median_pct_selection)} / {signed(x.d_median_pct_speed)}",
                     "H < 2 s per 100 events": num(e2.after_per100, 1) if obs else ci(e2.after_per100, e2.after_per100_lo95, e2.after_per100_hi95, 1),
                     "H < 1.5 s per 100 events": num(e15.after_per100, 1) if obs else ci(e15.after_per100, e15.after_per100_lo95, e15.after_per100_hi95, 1)})
add("T8_Speed_limits", "Table 8. BTW following under the e-rickshaw speed limits (S.R.O. No. 384-Law/2026)", pd.DataFrame(rows),
    "Limits: 25 km/h general (rule 2(gha)), 15 km/h near schools, hospitals, places of worship and crowded places (rule 8(4)); applied to BTW only. "
    "An event breaks up when the BTW is faster than the limit and follows a faster non-BTW leader; otherwise it follows at the limit with the same spacing "
    "(H' = H v / limit). Exposure per 100 observed BTW following events; events that break up are not exposed. Bootstrap (2,000 replicates).")
TABLES[-1][2]

### Table 9 — Dedicated BTW lane (step 6)

In [ ]:
dl = R["06_dedicated_btw_lane"]["scenarios"]
rows = []
for site in ["Shahjahanpur", "Tikatuli"]:
    for i, (_, x) in enumerate(dl[dl.Site == site].iterrows()):
        rows.append({"Site": site if i == 0 else "", "Scenario": x.Scenario, "n": int(x.n_following), "Median (s)": num(x.median_s, 2),
                     "H < 2 s (%, 95% CI)": pct_ci(x["share_H<2"], x["share_H<2_lo95"], x["share_H<2_hi95"]),
                     "H < 1.5 s (%, 95% CI)": pct_ci(x["share_H<1.5"], x["share_H<1.5_lo95"], x["share_H<1.5_hi95"])})
add("T9_Dedicated_lane", "Table 9. Short-headway exposure in mixed traffic and in a dedicated BTW lane at 25 km/h", pd.DataFrame(rows),
    "Dedicated lane: BTW followers behind BTW leaders, both limited to 25 km/h, so following never breaks up. The centred-only version uses the BTW-led "
    "events that were centred (small n at Shahjahanpur). Observed BTW-led following in mixed traffic stands in for in-lane behaviour.")
TABLES[-1][2]

### Appendix tables

In [ ]:
nl = R["01_truncated_distribution_fit"]["nested_LR"]
rows = []
for c in CASES:
    for i, (_, x) in enumerate(nl[(nl.Site == c[0]) & (nl.Class == c[1])].iterrows()):
        rows.append({"Case": CASE_LAB[c] if i == 0 else "", "Generalized gamma Q̂": num(x.GG_Q_hat, 3) if i == 0 else "",
                     "Restricted family": x.Restricted, "LR (1 df)": num(x.LR_stat, 2), "p": pval(x.p)})
add("A1_Nested_LR", "Table A1. Nested likelihood-ratio tests against the generalized gamma (unconditional fits)", pd.DataFrame(rows),
    "Lognormal: Q = 0; Weibull: Q = 1; gamma: Q = σ. A small p rejects the two-parameter family in favour of the generalized gamma.")

rows = []
for c in CASES:
    g = cand[(cand.Site == c[0]) & (cand.Class == c[1])].sort_values("AIC_leader")
    for i, (_, x) in enumerate(g.iterrows()):
        rows.append({"Case": CASE_LAB[c] if i == 0 else "", "Distribution": x.Family, "ΔAIC (within case)": num(x.dAIC_leader, 2),
                     "LR leader class (4 df) p": pval(x.LR_p), "KS p": gof(x.KS_p), "AD p": gof(x.AD_p), "CvM p": gof(x.CvM_p)})
add("A2_Leader_AFT_all_families", "Table A2. Leader-conditioned AFT fits for all five distributions", pd.DataFrame(rows),
    "Refitted parametric bootstrap, 999 replicates. In Shahjahanpur PR the generalized gamma sits on a weakly identified high-|Q| ridge.")

rows = []
for c in CASES:
    x = sp[(sp.Site == c[0]) & (sp.Class == c[1])].iloc[0]
    rows.append({"Case": CASE_LAB[c], "LR (4 df)": num(x.LR_stat, 2), "p": pval(x.LR_p), "ΔAIC (spread − shift only)": num(x.dAIC_spread_minus_shift, 2),
                 "σ shared": num(x.sigma_shared, 3), **{f"σ {L}": num(x[f"sigma_{L}"], 3) for L in LEADERS}})
add("A3_Shift_vs_spread", "Table A3. Does the leader change only the scale (shift) or also the spread of the headway distribution?", pd.DataFrame(rows),
    "Weibull AFT with leader-specific log σ against the shift-only model.")

pl = R["04_spacing_speed_decomposition"]["per_leader"]
rows = []
for site in ["Shahjahanpur", "Tikatuli"]:
    for i, (_, x) in enumerate(pl[pl.Site == site].iterrows()):
        rows.append({"Site": site if i == 0 else "", "Leader class": x.Leader, "n PR / BTW": f"{int(x.n_PR)} / {int(x.n_BTW)}",
                     "Headway PR ÷ BTW": num(x.headway_ratio, 2), "Spacing PR ÷ BTW": num(x.spacing_ratio, 2),
                     "Speed BTW ÷ PR": num(x.speed_ratio_BTW_over_PR, 2), "Speed share (%)": num(100 * x.speed_share, 0)})
add("A4_Decomposition_by_leader", "Table A4. Spacing–speed decomposition behind each leader class", pd.DataFrame(rows),
    "Geometric-mean ratios within each leader class; small cells, descriptive only.")

cl = R["05_speed_cap_scenarios"]["cases_by_leader"]
rows = []
for site in ["Shahjahanpur", "Tikatuli"]:
    for j, cap in enumerate([25, 15]):
        for i, (_, x) in enumerate(cl[(cl.Site == site) & (cl.Cap_kmh == cap)].iterrows()):
            rows.append({"Site": site if (i == 0 and j == 0) else "", "Speed limit": f"{cap} km/h" if i == 0 else "", "Leader class": x.Leader,
                         "n": int(x.n), "Not affected (%)": pct(x.share_case1), "Follows at the limit (%)": pct(x.share_case3),
                         "Following breaks up (%)": pct(x.share_case2)})
add("A5_Speed_limit_by_leader", "Table A5. Outcome of the speed limits for BTW followers by leader class", pd.DataFrame(rows),
    "Shares of the observed BTW following events behind each leader class.")

ss = R["05_speed_cap_scenarios"]["SP_BTW_leader_speed_sens"]; ls = R["06_dedicated_btw_lane"]["SP_leader_speed_sens"]
rows = []
for _, x in ss.iterrows():
    rows.append({"Scenario": f"Mixed traffic, {int(x.Cap_kmh)} km/h", "Without leader-speed effect": f"{num(x.median_s_no_leader_effect, 2)} s; "
                 f"{num(x.exp2_per100_no_leader_effect, 1)} / {num(x['exp1.5_per100_no_leader_effect'], 1)}",
                 "With leader-speed effect (TR 95% CI)": f"{num(x.median_s_TR_est, 2)} s ({num(x.median_s_TR_hi, 2)} to {num(x.median_s_TR_lo, 2)}); "
                 f"{num(x.exp2_per100_TR_est, 1)} / {num(x['exp1.5_per100_TR_est'], 1)}"})
for scen in ls.Scenario.unique():
    g = ls[ls.Scenario == scen].set_index("TR_leader_speed")
    rows.append({"Scenario": scen, "Without leader-speed effect": f"{num(g.loc['none', 'median_s'], 2)} s; {pct(g.loc['none', 'share_H<2'])}% / {pct(g.loc['none', 'share_H<1.5'])}%",
                 "With leader-speed effect (TR 95% CI)": f"{num(g.loc['est', 'median_s'], 2)} s ({num(g.loc['hi', 'median_s'], 2)} to {num(g.loc['lo', 'median_s'], 2)}); "
                 f"{pct(g.loc['est', 'share_H<2'])}% / {pct(g.loc['est', 'share_H<1.5'])}%"})
add("A6_SP_leader_speed_sensitivity", "Table A6. Shahjahanpur BTW: sensitivity to capped BTW leaders shortening the headway behind them", pd.DataFrame(rows),
    "Leader-adjusted time ratio 1.089 per 5 km/h of leader speed (notebook 03). Cells: median of the events still following; then short-headway "
    "exposure H < 2 s / H < 1.5 s (per 100 observed events for mixed traffic, share of lane events for the lane).")
print(len(TABLES), "tables")

### Appendix tables from the static appendix material

The pooling evidence, the traffic state and the detector metrics come from private inputs (data03, raw YOLO logs, videos). They are shipped as fixed results in `../appendix_static/` and copied into the workbook when present.

In [ ]:
pool_f, app_f = TAB / "Pooling_Tables.xlsx", TAB / "Appendix_Figure_Data.xlsx"   # written by notebook 1
VLAB = {"Time_Headway_s": "Time headway", "Target_Speed_kmh": "Follower speed", "Leading_Speed_kmh": "Leader speed"}
if pool_f.exists():
    P = pd.read_excel(pool_f, sheet_name=None)
    a = P["A_BTW_vs_ModBTW"]; rows = []
    for _, x in a.iterrows():
        rows.append({"Scope": x.scope, "Variable": VLAB[x.variable], "n BTW / modified BTW": f"{int(x.n_1)} / {int(x.n_2)}",
                     "Median BTW / modified BTW": f"{num(x.median_1, 2)} / {num(x.median_2, 2)}",
                     "Cliff's δ (95% CI)": ci(x.cliffs_delta, x.delta_CI_lo, x.delta_CI_hi), "KS p": pval(x.KS_p),
                     "TOST p (margin)": f"{pval(x.TOST_p)} (±{num(x.TOST_bound, 1)})", "BF01": num(x.BF01, 2)})
    add("A7_Target_pooling", "Table A7. Pooling of modified BTW with BTW as target vehicles", pd.DataFrame(rows),
        "Equivalence margins: ±0.3 s for headway, ±2 km/h for speed (TOST). BF01 > 3: evidence for no difference. Bootstrap CI of Cliff's δ.")
    om = P["B1_within_omnibus"]; rows = []
    for cls in om["class"].unique():
        g = om[om["class"] == cls].set_index("variable")
        rows.append({"Leader class": cls, "Pooled types": g.labels.iloc[0], "n": int(g.n.iloc[0]),
                     "Headway: KW p (ε²)": f"{pval(g.loc['Time_Headway_s', 'KW_p'])} ({num(g.loc['Time_Headway_s', 'epsilon2'], 3)})",
                     "Leader speed: KW p (ε²)": f"{pval(g.loc['Leading_Speed_kmh', 'KW_p'])} ({num(g.loc['Leading_Speed_kmh', 'epsilon2'], 3)})"})
    add("A8_Leader_pooling", "Table A8. Pooling of leader vehicle types into leader classes", pd.DataFrame(rows),
        "Kruskal–Wallis tests among the vehicle types pooled into each class. Pooling rests on the follower headway; bicycle leaders are faster "
        "than PR and van leaders, but the headway behind them does not differ. CNG and MT_2W kept as separate classes.")
if app_f.exists():
    A = pd.read_excel(app_f, sheet_name=None)
    ts = A["traffic_state_summary"]; rows = []
    for _, x in ts.iterrows():
        rows.append({"Session": x.Session, "Duration (min)": num(x.duration_min, 1), "Following events": int(x.n_events),
                     "Flow, mean ± SD (pcu/h)": f"{num(x.flow_mean_pcu_h, 0)} ± {num(x.flow_sd, 0)}",
                     "Flow, 5th–95th percentile (pcu/h)": f"{num(x.flow_p05, 0)} to {num(x.flow_p95, 0)}",
                     "3W share (%)": pct(x.mean_share_3W), "4W share (%)": pct(x.mean_share_4W), "Bus share (%)": pct(x.mean_share_Bus)})
    add("A9_Traffic_state", "Table A9. Traffic state during the recorded sessions", pd.DataFrame(rows),
        "Rolling 30-s centred flow from YOLO counts, converted with RHD (2005) PCU factors. SP and SP2 are the two Shahjahanpur sessions.")
    ym = A["yolo_metrics"]; rows = []
    for _, x in ym.iterrows():
        rows.append({"Class": x["class"], "AP50": num(x.AP50, 3), "AP50–95": num(x.AP50_95, 3),
                     "Precision": num(x.precision, 3), "Recall": num(x.recall, 3)})
    add("A10_Detector", "Table A10. Detector performance on the validation split (YOLO26n)", pd.DataFrame(rows),
        "Validation split only (the test folder duplicates it). Precision and recall reported for all classes combined.")
wsens = R["07_window_sensitivity"]
fw, tw, dw, nw = wsens["best_family"], wsens["weibull_leader_TR"], wsens["speed_share"], wsens["n_by_window"]
rows = []
for win in nw.Window:
    f_ = fw[fw.Window == win]; t_ = tw[tw.Window == win]; d_ = dw[dw.Window == win].set_index("Site")
    tr_ = lambda s, c, L: (lambda x: ci(x.TR, x.TR_lo95, x.TR_hi95, 2))(t_[(t_.Site == s) & (t_.Class == c) & (t_.Leader == L)].iloc[0])
    rows.append({"Window": win, "n (SP PR / SP BTW / TK PR / TK BTW)": " / ".join(str(int(nw[nw.Window == win][f"{s} {c}"].iloc[0])) for s, c in CASES),
                 "Weibull within 2 AIC of best (cases)": f"{int(f_.weibull_within_2.sum())} of 4",
                 "TR behind BTW, SP BTW": tr_("Shahjahanpur", "BTW", "BTW"), "TR behind BTW, TK BTW": tr_("Tikatuli", "BTW", "BTW"),
                 "TR behind NMT, TK BTW": tr_("Tikatuli", "BTW", "NMT"),
                 "Speed share, SP (%)": pct_ci(d_.loc["Shahjahanpur", "speed_share"], d_.loc["Shahjahanpur", "speed_share_lo95"], d_.loc["Shahjahanpur", "speed_share_hi95"], 0),
                 "Speed share, TK (%)": pct_ci(d_.loc["Tikatuli", "speed_share"], d_.loc["Tikatuli", "speed_share_lo95"], d_.loc["Tikatuli", "speed_share_hi95"], 0)})
add("A11_Window_sensitivity", "Table A11. Sensitivity of the main results to the 0.5–5 s extraction window", pd.DataFrame(rows),
    "Each narrower window is refitted with the likelihood truncated to its own bounds. TR = Weibull AFT time ratio vs a 4W leader (95% Wald CI). "
    "Speed share = share of the PR–BTW log-headway contrast due to speed (stratified bootstrap, 1,000 replicates).")
print(len(TABLES), "tables in total")

### Figure list

In [ ]:
FIGS = [
    ("truncated_fit_density", "Truncated distribution fits by site and target class", "01"),
    ("truncated_fit_cdf_difference", "Empirical minus fitted CDF for each distribution", "01"),
    ("leader_aft_time_ratios", "Time ratios by leader class, Weibull AFT", "02"),
    ("leader_aft_cox_snell", "Cox–Snell residuals of the leader-conditioned Weibull AFT", "02"),
    ("leader_shifted_densities", "Leader-specific headway distributions against the main distribution", "02b"),
    ("leader_shift_vs_main", "Shift of the median and of short headways by leader class", "02b"),
    ("covariate_time_ratios", "Time ratios of off-centred following, occupancy and leader speed", "03"),
    ("decomposition_contrast", "PR–BTW headway contrast split into spacing and speed", "04"),
    ("spacing_speed_scatter", "Spacing against follower speed with constant-headway lines", "04"),
    ("speed_cap_outcome_shares", "Outcome of the speed limits for BTW following events", "05"),
    ("speed_cap_headway_ecdf", "Headways of BTWs still following under the speed limits", "05"),
    ("speed_cap_short_headway_exposure", "Exposure to short headways under the speed limits", "05"),
    ("dedicated_lane_exposure", "Short-headway exposure in mixed traffic and in a dedicated BTW lane", "06"),
    ("dedicated_lane_headway_ecdf", "Headway distribution in mixed traffic and in a dedicated BTW lane", "06"),
    ("window_sensitivity", "Leader time ratios and spacing–speed split across extraction windows", "07"),
]
fig_tab = pd.DataFrame([{"File": f"{f}.png", "Content": c, "Notebook": nb, "Present": "yes" if (FIG / f"{f}.png").exists() else "MISSING"}
                        for f, c, nb in FIGS])
APPX = [("traffic_state_rolling_flow.png", "Rolling 30-s flow during each session"),
        ("event_timing.png", "Timing of the following events within each session"), ("yolo_detector_performance.png", "Detector performance (validation)"),
        ("target_subtypes_ECDF.png", "Headway of BTW and modified BTW targets"), ("leader_speed_by_label.png", "Leader speed by vehicle type"),
        ("headway_by_leader_label.png", "Follower headway by leader vehicle type")]
appx_tab = pd.DataFrame([{"File": f, "Content": c, "Notebook": "1 (appendix)",
                          "Present": "yes" if (FIG / f).exists() else "MISSING"} for f, c in APPX])
fig_tab = pd.concat([fig_tab, appx_tab], ignore_index=True)
print(fig_tab.to_string(index=False))

### Write the workbook

In [ ]:
FONT = "Cambria"; thin = Side(style="thin", color="000000")
wb = Workbook(); ix = wb.active; ix.title = "Index"
ix.append(["Sheet", "Title"])
for sheet, title, df, notes in TABLES:
    ix.append([sheet, title])
ix.append([]); ix.append(["Figures (PNG, 600 dpi)", ""])
ix.append(["File", "Content", "Notebook"])
for _, r in fig_tab.iterrows():
    ix.append([r.File, r.Content, r.Notebook])
for sheet, title, df, notes in TABLES:
    ws = wb.create_sheet(sheet)
    ws.append([title]); ws.append(list(df.columns))
    for row in df.itertuples(index=False):
        ws.append(list(row))
    ws.append([]); ws.append(["Note: " + notes])
    ncol = len(df.columns); last = 2 + len(df)
    for row in ws.iter_rows():
        for c in row:
            c.font = Font(name=FONT, size=11, bold=(c.row in (1, 2)))
            c.alignment = Alignment(vertical="top", wrap_text=(c.row == last + 2))
    for j in range(1, ncol + 1):
        ws.cell(2, j).border = Border(top=thin, bottom=thin); ws.cell(last, j).border = Border(bottom=thin)
        width = max(len(str(ws.cell(i, j).value or "")) for i in range(2, last + 1))
        ws.column_dimensions[get_column_letter(j)].width = min(max(10, width + 2), 48)
    ws.merge_cells(start_row=last + 2, start_column=1, end_row=last + 2, end_column=ncol)
    ws.row_dimensions[last + 2].height = 15 * (1 + len(notes) // 120)
    ws.freeze_panes = "A3"
for row in ix.iter_rows():
    for c in row:
        c.font = Font(name=FONT, size=11, bold=(c.row == 1 or c.value in ("File", "Figures (PNG, 600 dpi)")))
ix.column_dimensions["A"].width = 38; ix.column_dimensions["B"].width = 100; ix.column_dimensions["C"].width = 10
OUT.unlink(missing_ok=True); wb.save(OUT); print("wrote", OUT.name, "with", len(TABLES), "tables")